# LeetCode 403 — Frog Jump bằng Python (Hard)

Notebook này giảng nhập môn bài toán:

```python
class Solution:
    def canCross(self, stones: List[int]) -> bool:

```

Mục tiêu của bài học:

1. Hiểu đề bằng **hình vẽ con sông**: đá ở đâu, nước ở đâu, con ếch được nhảy bao xa. Nhận ra điểm "khó" của bài: **bước nhảy tiếp theo phụ thuộc bước nhảy vừa rồi**, nên chỉ biết "ếch đang ở đâu" là **chưa đủ**: ta cần **trạng thái** (state) `(vị trí, bước cuối)`.
2. Học từ đầu các công cụ Python cần dùng: `set` (tra cứu nhanh), `tuple` làm khóa của `dict`, **đệ quy** (recursion), **memoization** (ghi nhớ kết quả) bằng `dict` và `functools.lru_cache`, giới hạn đệ quy `sys.setrecursionlimit`, hàng đợi `deque`.
3. Viết lời giải **đệ quy thử 3 bước `k-1, k, k+1`** (brute force) và đo tận mắt nó chạy theo **hàm mũ**.
4. Sửa bằng **đệ quy + memo theo `(vị trí, k)`**: cách chính kiểu top-down, `O(n²)`.
5. Giải lại bằng **DP bottom-up "lan tỏa"**: `dict` *đá → tập các bước nhảy đã tới được*, trace đầy đủ ví dụ `[0,1,3,5,6,8,12,17]`.
6. Giải bằng **BFS trên trạng thái** `(đá, k)`: mỗi "lớp" BFS = một lần nhảy.
7. Hiểu và chứng minh mẹo **cắt tỉa**: *bước nhảy thứ `t` luôn `≤ t`*, nên nếu `stones[i] - stones[i-1] > i` thì trả `False` ngay.
8. Kiểm thử chéo mọi cách, tránh các lỗi kinh điển (quên `step > 0`, sai bước nhảy đầu tiên, memo chỉ theo vị trí, tràn đệ quy...) và nhận dạng pattern "DP trên trạng thái (vị trí, hành động trước)".

## 1. Đề bài

Đề LeetCode (dịch sát nghĩa):

> Một con ếch đang băng qua một con sông. Con sông được chia thành nhiều **đơn vị** (unit), tại mỗi đơn vị **có thể có hoặc không có** một hòn đá. Con ếch được nhảy lên đá, nhưng **không được nhảy xuống nước**.
>
> Cho danh sách `stones` là **vị trí các hòn đá** (tính theo đơn vị), đã **sắp xếp tăng dần**. Hãy xác định con ếch có thể băng qua sông bằng cách **đáp xuống hòn đá cuối cùng** hay không.
>
> Ban đầu con ếch đứng trên hòn đá đầu tiên, và **bước nhảy đầu tiên bắt buộc dài đúng 1 đơn vị**.
>
> Nếu bước nhảy vừa rồi của ếch dài `k` đơn vị, thì bước nhảy tiếp theo phải dài **`k - 1`, `k` hoặc `k + 1`** đơn vị. Con ếch **chỉ nhảy về phía trước**.

**Example 1**

```text
Input: stones = [0,1,3,5,6,8,12,17]
Output: true
Giải thích: Ếch nhảy 1 đơn vị tới đá thứ 2, rồi 2 đơn vị tới đá thứ 3, rồi 2 đơn vị tới đá thứ 4,
rồi 3 đơn vị tới đá thứ 6, 4 đơn vị tới đá thứ 7, và 5 đơn vị tới đá thứ 8 (đá cuối).
```

**Example 2**

```text
Input: stones = [0,1,2,3,4,8,9,11]
Output: false
Giải thích: Không có cách nào nhảy tới đá cuối, vì khoảng trống giữa đá thứ 5 và đá thứ 6 quá lớn.
```

**Constraints**

- `2 <= stones.length <= 2000`
- `0 <= stones[i] <= 2^31 - 1`
- `stones[0] == 0`
- `stones` được sắp xếp **tăng ngặt** (strictly increasing, không có hai đá trùng vị trí).

### Ký hiệu dùng suốt notebook

- `n = len(stones)`: số hòn đá (tối đa 2000).
- "Đá thứ `i`" theo **chỉ số Python** (đếm từ 0): `stones[i]` là **vị trí** của nó. Ví dụ ở Example 1, đá chỉ số `5` nằm ở vị trí `8`. (Đề đếm "đá thứ 2, thứ 3..." từ 1; notebook này đếm chỉ số từ 0 như Python.)
- `k`: độ dài **bước nhảy vừa thực hiện** để đáp xuống hòn đá hiện tại.
- `last = stones[-1]`: vị trí hòn đá cuối, đích đến.

### 1.1 Hiểu đề bằng hình: vẽ con sông

Vẽ Example 1 trên trục số, `#` là đá, `~` là nước:

```text
vị trí: 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17
sông  : # # ~ # ~ # # ~ # ~ ~  ~  #  ~  ~  ~  ~  #
```

Đường đi của đề: `0 →(1) 1 →(2) 3 →(2) 5 →(3) 8 →(4) 12 →(5) 17`. Số trong ngoặc là độ dài bước nhảy. Kiểm tra luật từng bước:

| Bước | Từ → tới | Độ dài | Bước trước `k` | Được phép `k-1, k, k+1` | Hợp lệ? |
|---:|---|---:|---:|---|:---:|
| 1 | `0 → 1` | 1 | (đầu tiên) | bắt buộc `1` | ✅ |
| 2 | `1 → 3` | 2 | 1 | `0, 1, 2` | ✅ |
| 3 | `3 → 5` | 2 | 2 | `1, 2, 3` | ✅ |
| 4 | `5 → 8` | 3 | 2 | `1, 2, 3` | ✅ |
| 5 | `8 → 12` | 4 | 3 | `2, 3, 4` | ✅ |
| 6 | `12 → 17` | 5 | 4 | `3, 4, 5` | ✅ |

Vài suy luận nhỏ mà rất quan trọng:

- **Bước nhảy độ dài `0` vô nghĩa** (đứng yên tại chỗ) và đề cấm nhảy lùi, nên chỉ xét các bước **`> 0`**. Ví dụ `k = 1` thì `k - 1 = 0` bị loại, chỉ còn `1, 2`.
- Ếch **không bắt buộc** phải dừng ở mọi hòn đá: ở bước 4 nó nhảy từ `5` thẳng qua `6` để tới `8`.
- Đích là **đáp xuống đá cuối**; nhảy "vượt qua" đá cuối không tính (và cũng không có đá nào sau nó).
- Vị trí đá có thể tới `2^31 - 1` (rất lớn), nhưng chỉ có tối đa 2000 hòn. Vì vậy ta **không** tạo mảng dài theo vị trí (2 tỉ ô!) mà làm việc với **tập các vị trí đá**.

Cell dưới vẽ con sông cho cả 2 ví dụ và **kiểm tra một chuỗi bước nhảy có hợp lệ không**, để xác nhận bảng trên.

In [1]:
from typing import List
from functools import lru_cache
from collections import deque
import random
import sys
import time

EX1 = [0, 1, 3, 5, 6, 8, 12, 17]
EX2 = [0, 1, 2, 3, 4, 8, 9, 11]


def draw_river(stones):
    # In con sông: '#' là đá, '~' là nước. Chỉ dùng cho ví dụ nhỏ!
    stone_set = set(stones)
    last = stones[-1]
    print("vị trí:", " ".join(f"{p:>2}" for p in range(last + 1)))
    print("sông  :", " ".join(f"{'#' if p in stone_set else '~':>2}" for p in range(last + 1)))


def is_valid_jumps(stones, jumps):
    # jumps: danh sách độ dài các bước nhảy, xuất phát từ vị trí 0.
    stone_set = set(stones)
    pos, k = 0, None
    for step in jumps:
        if k is None and step != 1:                 # bước đầu tiên phải là 1
            return False
        if k is not None and step not in (k - 1, k, k + 1):
            return False
        if step <= 0:                               # không đứng yên, không nhảy lùi
            return False
        pos += step
        if pos not in stone_set:                    # rơi xuống nước
            return False
        k = step
    return pos == stones[-1]                        # phải đáp đúng đá cuối


draw_river(EX1)
print("đường đi của đề [1,2,2,3,4,5]:", is_valid_jumps(EX1, [1, 2, 2, 3, 4, 5]))
print("bước đầu dài 2           [2]:", is_valid_jumps(EX1, [2]))
print("1 -> 2 là nước        [1, 1]:", is_valid_jumps(EX1, [1, 1]))
print("đổi bước quá nhiều [1,2,2,3,5]:", is_valid_jumps(EX1, [1, 2, 2, 3, 5]))
print()
draw_river(EX2)

vị trí:  0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17
sông  :  #  #  ~  #  ~  #  #  ~  #  ~  ~  ~  #  ~  ~  ~  ~  #
đường đi của đề [1,2,2,3,4,5]: True
bước đầu dài 2           [2]: False
1 -> 2 là nước        [1, 1]: False
đổi bước quá nhiều [1,2,2,3,5]: False

vị trí:  0  1  2  3  4  5  6  7  8  9 10 11
sông  :  #  #  #  #  #  ~  ~  ~  #  #  ~  #


### 1.2 Vì sao Example 2 thất bại?

Nhìn con sông của Example 2: các đá `0, 1, 2, 3, 4` sát nhau, rồi tới **khoảng trống** từ `4` sang `8` (dài 4 đơn vị).

Muốn qua khoảng trống đó, ếch phải nhảy **ít nhất 4** đơn vị trong một lần, mà muốn nhảy 4 thì bước trước đó phải là `3, 4 hoặc 5`. Nhưng trên dãy đá `0..4` (mỗi đá cách nhau 1), bước nhảy chỉ tăng được chậm: `1 → 2 → ...`, và chỉ có 4 đơn vị đường để "lấy đà". Ta sẽ thấy ở mục DP rằng khi đứng ở đá `4`, bước nhảy lớn nhất ếch có thể vừa thực hiện là `2`, nên bước tiếp theo tối đa là `3` và rơi vào vị trí `7`, là nước.

Để chuẩn bị, hãy tính **khoảng cách giữa hai đá liền kề** (gap). Đây là con số ta sẽ dùng lại ở mẹo cắt tỉa cuối bài. `zip(stones, stones[1:])` ghép mỗi đá với đá ngay sau nó.

In [2]:
def gaps(stones):
    # Khoảng cách giữa đá i-1 và đá i, với i = 1..n-1
    return [b - a for a, b in zip(stones, stones[1:])]

print("zip ghép cặp liền kề:", list(zip(EX2, EX2[1:]))[:4], "...")
print("gaps EX1:", gaps(EX1))
print("gaps EX2:", gaps(EX2))

zip ghép cặp liền kề: [(0, 1), (1, 2), (2, 3), (3, 4)] ...
gaps EX1: [1, 2, 2, 1, 2, 4, 5]
gaps EX2: [1, 1, 1, 1, 4, 1, 2]


## 2. Khái niệm cốt lõi: TRẠNG THÁI `(vị trí, k)`

Ở nhiều bài "đi trên mảng" (như *Climbing Stairs*), chỉ cần biết **đang đứng ở đâu** là đủ để biết bước tiếp theo đi được đâu. Bài này thì **không**.

Xét hòn đá ở vị trí `8` trong Example 1. Ếch có thể tới đó theo 2 cách:

| Đến `8` bằng | Bước cuối `k` | Bước tiếp được phép | Vị trí tới | Kết quả |
|---|---:|---|---|---|
| `6 → 8` | 2 | `1, 2, 3` | `9, 10, 11` | toàn nước → **kẹt** |
| `5 → 8` | 3 | `2, 3, 4` | `10, 11, 12` | `12` là đá → **đi tiếp được** |

Cùng một hòn đá, nhưng **tương lai khác nhau hoàn toàn** tùy bước nhảy cuối. Vì vậy, mô tả đầy đủ "con ếch đang ở tình huống nào" cần **hai** con số:

```text
trạng thái = (pos, k)   # pos: vị trí đang đứng, k: độ dài bước vừa nhảy
```

Từ trạng thái `(pos, k)`, các bước nhảy tiếp có thể là `k-1, k, k+1` (chỉ giữ cái `> 0`).

**Mẹo khởi đầu:** bước đầu tiên bắt buộc là `1`. Nếu ta giả vờ ếch "vừa nhảy bước `0`" để tới đá `0`, thì `k-1, k, k+1 = -1, 0, 1`, lọc `> 0` còn đúng `[1]`. Vậy **trạng thái xuất phát là `(0, 0)`** và ta không cần viết riêng trường hợp bước đầu tiên. Mẹo nhỏ nhưng giúp code gọn, ta sẽ dùng ở mọi cách giải.

In [3]:
def next_steps(k):
    # Các độ dài bước nhảy hợp lệ sau khi vừa nhảy bước k.
    return [step for step in (k - 1, k, k + 1) if step > 0]

print("k = 0 (xuất phát):", next_steps(0))   # chỉ còn [1] -> đúng luật bước đầu tiên
print("k = 1            :", next_steps(1))   # 0 bị loại
print("k = 2            :", next_steps(2))
print("k = 3            :", next_steps(3))

stone_set = set(EX1)
for k in (2, 3):
    landing = [(8 + s, "đá" if 8 + s in stone_set else "nước") for s in next_steps(k)]
    print(f"đứng ở 8 với k = {k}:", landing)

k = 0 (xuất phát): [1]
k = 1            : [1, 2]
k = 2            : [1, 2, 3]
k = 3            : [2, 3, 4]
đứng ở 8 với k = 2: [(9, 'nước'), (10, 'nước'), (11, 'nước')]
đứng ở 8 với k = 3: [(10, 'nước'), (11, 'nước'), (12, 'đá')]


## 3. Nền tảng Python

### 3.1 `set`: tra cứu "có đá ở vị trí này không?" thật nhanh

Thao tác ta làm nhiều nhất là hỏi: *"vị trí `pos + step` có phải là đá không?"*.

- Nếu dùng `list`, câu hỏi `x in stones` phải **dò từng phần tử** từ đầu: tốn `O(n)`.
- Nếu đổi sang `set` (tập hợp), Python dùng **bảng băm** (hash table), câu hỏi `x in stone_set` trung bình chỉ tốn `O(1)`.

```python
stone_set = set(stones)    # tạo một lần, O(n)
12 in stone_set            # True / False, trung bình O(1)
```

`set` không giữ thứ tự và không có phần tử trùng. Thêm phần tử bằng `.add(x)`; thêm phần tử đã có thì **không có gì thay đổi** (tính chất này rất quan trọng ở cách DP bottom-up).

In [4]:
stone_set = set(EX1)
print("stone_set     :", stone_set)
print("12 in set     :", 12 in stone_set)
print("13 in set     :", 13 in stone_set)

jumps_at_8 = set()
jumps_at_8.add(2)
jumps_at_8.add(3)
jumps_at_8.add(2)            # thêm trùng: không đổi gì
print("jumps_at_8    :", jumps_at_8, "| số phần tử:", len(jumps_at_8))

# So sánh tốc độ tra cứu list và set trên 200 000 phần tử
big_list = list(range(0, 400000, 2))
big_set = set(big_list)
t0 = time.perf_counter(); _ = [399998 in big_list for _ in range(200)]; t1 = time.perf_counter()
_ = [399998 in big_set for _ in range(200)]; t2 = time.perf_counter()
print(f"200 lần tra trong list: {t1 - t0:.4f}s | trong set: {t2 - t1:.6f}s")

stone_set     : {0, 1, 3, 5, 6, 8, 12, 17}
12 in set     : True
13 in set     : False
jumps_at_8    : {2, 3} | số phần tử: 2
200 lần tra trong list: 0.1717s | trong set: 0.000079s


### 3.2 `tuple` làm khóa (key) của `dict`

Ta sẽ cần một "cuốn sổ" ghi: *trạng thái `(pos, k)` đã biết kết quả là True hay False*. `dict` (từ điển) lưu cặp `key → value`, và key có thể là một **tuple** `(pos, k)`.

- Tuple là dãy giá trị **không thay đổi được** (immutable), viết trong ngoặc tròn: `(8, 3)`.
- Chỉ những thứ không đổi được (số, chuỗi, tuple...) mới làm key được. `list` như `[8, 3]` thì **không** làm key được (Python báo `TypeError: unhashable type`), vì list có thể bị sửa sau khi đã cất vào bảng băm.

In [5]:
memo = {}
memo[(8, 2)] = False          # đứng ở 8 với k=2: kẹt
memo[(8, 3)] = True           # đứng ở 8 với k=3: đi tiếp được
print("memo         :", memo)
print("(8, 3) in memo:", (8, 3) in memo, "->", memo[(8, 3)])
print("(8, 4) in memo:", (8, 4) in memo)

try:
    memo[[8, 5]] = True       # list làm key -> lỗi
except TypeError as e:
    print("Dùng list làm key:", e)

memo         : {(8, 2): False, (8, 3): True}
(8, 3) in memo: True -> True
(8, 4) in memo: False
Dùng list làm key: unhashable type: 'list'


### 3.3 Đệ quy (recursion) từ con số 0

**Đệ quy** là khi một hàm **tự gọi lại chính nó** với một bài toán **nhỏ hơn**. Mọi hàm đệ quy đúng đều có 2 phần:

1. **Base case** (trường hợp cơ sở): bài toán đủ nhỏ để trả lời ngay, **không** gọi tiếp. Thiếu phần này hàm sẽ gọi mãi không dừng.
2. **Recursive case** (trường hợp đệ quy): chia bài toán thành bài nhỏ hơn, gọi lại chính hàm đó, rồi dùng kết quả.

Ví dụ mini: tính `1 + 2 + ... + n`.

```text
sum_to(4) = 4 + sum_to(3)
          = 4 + 3 + sum_to(2)
          = 4 + 3 + 2 + sum_to(1)
          = 4 + 3 + 2 + 1 + sum_to(0)     <- base case: sum_to(0) = 0
          = 10
```

Với Frog Jump, câu hỏi "từ trạng thái `(pos, k)` có tới được đá cuối không?" cũng chia nhỏ tự nhiên: *"có, nếu **một trong** các trạng thái kế tiếp tới được đá cuối"*.

In [6]:
def sum_to(n, depth=0):
    print("  " * depth + f"gọi sum_to({n})")
    if n == 0:                        # base case
        return 0
    result = n + sum_to(n - 1, depth + 1)   # recursive case: bài nhỏ hơn
    print("  " * depth + f"sum_to({n}) = {result}")
    return result

print("Kết quả:", sum_to(4))

gọi sum_to(4)
  gọi sum_to(3)
    gọi sum_to(2)
      gọi sum_to(1)
        gọi sum_to(0)
      sum_to(1) = 1
    sum_to(2) = 3
  sum_to(3) = 6
sum_to(4) = 10
Kết quả: 10


### 3.4 Cây gọi đệ quy có thể "nổ" vì tính lại bài con trùng nhau

Khi một lời gọi sinh ra **nhiều** lời gọi con, các lời gọi tạo thành một **cây**. Ví dụ kinh điển: Fibonacci `fib(n) = fib(n-1) + fib(n-2)`.

```text
                fib(5)
              /        \
          fib(4)        fib(3)        <- fib(3) bị tính 2 lần
         /     \        /    \
     fib(3)  fib(2)  fib(2) fib(1)    <- fib(2) bị tính 3 lần
      ...
```

Cùng một bài con (`fib(3)`, `fib(2)`) bị tính đi tính lại, số lời gọi tăng theo **hàm mũ**. Đếm thử:

In [7]:
calls = 0

def fib_naive(n):
    global calls
    calls += 1
    if n <= 1:
        return n
    return fib_naive(n - 1) + fib_naive(n - 2)

for n in (10, 15, 20, 25):
    calls = 0
    value = fib_naive(n)
    print(f"fib({n}) = {value:>6} | số lời gọi = {calls:>7}")

fib(10) =     55 | số lời gọi =     177
fib(15) =    610 | số lời gọi =    1973
fib(20) =   6765 | số lời gọi =   21891
fib(25) =  75025 | số lời gọi =  242785


### 3.5 Memoization: ghi sổ để không tính lại

**Memoization** (ghi nhớ) = lần đầu tính xong một bài con thì **ghi kết quả vào sổ** (`dict`); lần sau gặp lại thì **đọc sổ** thay vì tính lại. Mỗi bài con chỉ tính **một lần**.

Có 2 cách viết trong Python:

1. **Tự quản lý `dict`**: rõ ràng, dễ trace, dễ hiểu bản chất.
2. **Decorator `@lru_cache(maxsize=None)`** từ `functools`: đặt trên đầu hàm, Python tự ghi sổ theo **bộ tham số** của hàm. Ngắn gọn, nhưng tham số phải "hash được" (số, chuỗi, tuple; không phải list). `.cache_info()` cho biết số lần trúng sổ (`hits`), số lần phải tính (`misses`).

In [8]:
def fib_memo(n, memo):
    if n in memo:                     # đã có trong sổ -> dùng luôn
        return memo[n]
    if n <= 1:
        return n
    memo[n] = fib_memo(n - 1, memo) + fib_memo(n - 2, memo)   # tính xong -> ghi sổ
    return memo[n]

print("fib_memo(80) =", fib_memo(80, {}))


@lru_cache(maxsize=None)
def fib_cached(n):
    if n <= 1:
        return n
    return fib_cached(n - 1) + fib_cached(n - 2)

print("fib_cached(80) =", fib_cached(80))
print(fib_cached.cache_info())        # misses = 81 bài con, mỗi bài tính đúng 1 lần

fib_memo(80) = 23416728348467685
fib_cached(80) = 23416728348467685
CacheInfo(hits=78, misses=81, maxsize=None, currsize=81)


### 3.6 Giới hạn độ sâu đệ quy

Mỗi lời gọi hàm chưa kết thúc chiếm một "tầng" trên **call stack** (ngăn xếp lời gọi). Python mặc định chỉ cho khoảng **1000 tầng**; vượt quá sẽ báo `RecursionError`.

Ở Frog Jump, chuỗi nhảy dài nhất có thể đi qua **cả 2000 hòn đá**, tức đệ quy sâu tới ~2000 tầng, **vượt mức mặc định**. Cách xử lý:

- Nâng giới hạn: `sys.setrecursionlimit(10000)` (dùng ở bản top-down).
- Hoặc tránh hẳn đệ quy: dùng DP bottom-up / BFS (mục 7, 8).

Cell dưới **nâng giới hạn lên 10000 cho toàn bộ notebook** (các cell sau dựa vào điều này).

In [9]:
def depth(n):
    return 0 if n == 0 else 1 + depth(n - 1)

print("Giới hạn mặc định:", sys.getrecursionlimit())
try:
    depth(3000)
except RecursionError as e:
    print("depth(3000) ->", type(e).__name__, ":", e)

sys.setrecursionlimit(10000)
print("Giới hạn mới     :", sys.getrecursionlimit())
print("depth(3000)      =", depth(3000))

Giới hạn mặc định: 1000
depth(3000) -> RecursionError : maximum recursion depth exceeded
Giới hạn mới     : 10000
depth(3000)      = 3000


## 4. Cách 1 — Đệ quy thử cả 3 bước (brute force)

Đây là cách mà người mới tự nghĩ ra được: **mô phỏng con ếch**, ở mỗi hòn đá thử lần lượt 3 bước, bước nào đáp xuống đá thì "đi theo" bước đó; nếu đường đó thất bại thì **quay lui** (backtrack) thử bước khác.

Định nghĩa hàm:

```text
can(pos, k) = True nếu đang ở vị trí pos, vừa nhảy bước k, có thể tới đá cuối.
```

- **Base case:** `pos == last` → `True` (đã đứng trên đá cuối).
- **Recursive case:** với mỗi `step` trong `k-1, k, k+1`:
  - bỏ qua nếu `step <= 0`;
  - bỏ qua nếu `pos + step` không phải đá (rơi xuống nước);
  - nếu `can(pos + step, step)` là `True` → trả `True` ngay.
- Thử hết mà không được → `False`.
- Đáp án của bài = `can(0, 0)` (mẹo `k = 0` ở mục 2).

Hàm này **chắc chắn dừng**: mỗi lời gọi con có `pos` **lớn hơn** hẳn (vì `step > 0`), mà vị trí không thể vượt `last` mãi.

In [10]:
def can_cross_bruteforce(stones: List[int]) -> bool:
    stone_set = set(stones)
    last = stones[-1]

    def can(pos, k):
        if pos == last:                      # base case: đứng trên đá cuối
            return True
        for step in (k - 1, k, k + 1):
            if step > 0 and pos + step in stone_set:
                if can(pos + step, step):    # đi thử đường này
                    return True
        return False                         # thử hết 3 bước đều thất bại

    return can(0, 0)


print("EX1:", can_cross_bruteforce(EX1))     # True
print("EX2:", can_cross_bruteforce(EX2))     # False
print("[0, 1]:", can_cross_bruteforce([0, 1]))   # True: nhảy 1 là tới
print("[0, 2]:", can_cross_bruteforce([0, 2]))   # False: bước đầu phải là 1

EX1: True
EX2: False
[0, 1]: True
[0, 2]: False


### 4.1 Trace: nhìn con ếch thử và quay lui

Hàm `trace_bruteforce` in mỗi lời gọi, thụt lề theo độ sâu. Chạy trên Example 1, chú ý đoạn giữa:

- Từ `can(5, 2)`, bước `1` đưa ếch tới `(6, 1)`, rồi `(8, 2)`. Ở `(8, 2)` cả 3 vị trí `9, 10, 11` đều là nước → `False`, **quay lui** về `(6, 1)` rồi về `(5, 2)`.
- `(5, 2)` thử tiếp bước `2` (vị trí `7`, nước), rồi bước `3` → `(8, 3)` → `(12, 4)` → `(17, 5)`: tới đích.

Đây đúng là tình huống ở mục 2: cùng đá `8`, `k = 2` thì kẹt, `k = 3` thì đi được.

In [11]:
def trace_bruteforce(stones):
    stone_set = set(stones)
    last = stones[-1]

    def can(pos, k, depth):
        ind = "  " * depth
        if pos == last:
            print(f"{ind}can({pos}, {k}) -> tới đá cuối: True")
            return True
        print(f"{ind}can({pos}, {k}) thử bước {next_steps(k)}")
        for step in next_steps(k):
            nxt = pos + step
            if nxt not in stone_set:
                print(f"{ind}  bước {step}: {nxt} là nước")
                continue
            if can(nxt, step, depth + 1):
                return True
        print(f"{ind}=> can({pos}, {k}) = False (quay lui)")
        return False

    return can(0, 0, 0)

print("Kết quả:", trace_bruteforce(EX1))

can(0, 0) thử bước [1]
  can(1, 1) thử bước [1, 2]
    bước 1: 2 là nước
    can(3, 2) thử bước [1, 2, 3]
      bước 1: 4 là nước
      can(5, 2) thử bước [1, 2, 3]
        can(6, 1) thử bước [1, 2]
          bước 1: 7 là nước
          can(8, 2) thử bước [1, 2, 3]
            bước 1: 9 là nước
            bước 2: 10 là nước
            bước 3: 11 là nước
          => can(8, 2) = False (quay lui)
        => can(6, 1) = False (quay lui)
        bước 2: 7 là nước
        can(8, 3) thử bước [2, 3, 4]
          bước 2: 10 là nước
          bước 3: 11 là nước
          can(12, 4) thử bước [3, 4, 5]
            bước 3: 15 là nước
            bước 4: 16 là nước
            can(17, 5) -> tới đá cuối: True
Kết quả: True


### 4.2 Nút thắt: số lời gọi tăng theo hàm mũ

Mỗi lời gọi có thể sinh ra **tới 3** lời gọi con. Khi các hòn đá nằm sát nhau (nhiều lựa chọn hợp lệ) và đáp án là `False` (phải thử **hết** mọi đường mới dám kết luận), cây gọi phình ra rất nhanh.

Thí nghiệm: đá ở `0, 1, 2, ..., n-1` (sát nhau) cộng thêm một đá cuối ở vị trí `2n - 2`, quá xa để tới được. Đếm số lời gọi:

In [12]:
def bruteforce_calls(stones):
    # Giống can_cross_bruteforce nhưng đếm số lời gọi hàm can.
    stone_set = set(stones)
    last = stones[-1]
    calls = 0

    def can(pos, k):
        nonlocal calls
        calls += 1
        if pos == last:
            return True
        for step in (k - 1, k, k + 1):
            if step > 0 and pos + step in stone_set and can(pos + step, step):
                return True
        return False

    return can(0, 0), calls


print(" n đá | kết quả | số lời gọi")
for m in range(8, 23, 2):
    stones = list(range(m)) + [2 * m - 2]
    result, calls = bruteforce_calls(stones)
    print(f"{len(stones):>5} | {str(result):>7} | {calls:>9}")

 n đá | kết quả | số lời gọi
    9 |   False |        36
   11 |   False |       100
   13 |   False |       279
   15 |   False |       777
   17 |   False |      2165
   19 |   False |      6031
   21 |   False |     16795
   23 |   False |     46759


Mỗi khi thêm 2 hòn đá, số lời gọi nhân lên gần **2.8 lần**, tức khoảng `1.67` lần cho mỗi hòn đá: đúng dáng **hàm mũ** `O(c^n)`. Với `n = 2000` thì con số này lớn hơn số nguyên tử trong vũ trụ. Brute force **không thể** qua được LeetCode.

Câu hỏi then chốt: *trong mười mấy nghìn lời gọi đó, có bao nhiêu trạng thái `(pos, k)` thật sự khác nhau?*

## 5. Bước đột phá: trạng thái lặp lại rất nhiều

Kết quả của `can(pos, k)` **chỉ phụ thuộc vào `pos` và `k`**, không phụ thuộc ếch đã đi đường nào để tới đó. Ví dụ đá `0, 1, 2, 3, ...`: ếch có thể tới `(3, 1)` qua đường `0→1→2→3`, hoặc tới `(3, 2)` qua `0→1→3`... và rất nhiều đường khác nhau cùng dẫn tới **một** trạng thái như `(5, 2)`. Brute force tính lại toàn bộ cây con của `(5, 2)` **mỗi lần** gặp nó.

Đây chính là dấu hiệu kinh điển của **quy hoạch động** (Dynamic Programming, DP): *bài con trùng nhau* (overlapping subproblems). Hãy đếm số trạng thái **khác nhau** so với số lời gọi:

In [13]:
def distinct_states(stones):
    # Đếm số trạng thái (pos, k) khác nhau mà brute force từng ghé.
    stone_set = set(stones)
    last = stones[-1]
    seen = set()

    def can(pos, k):
        seen.add((pos, k))
        if pos == last:
            return True
        for step in (k - 1, k, k + 1):
            if step > 0 and pos + step in stone_set and can(pos + step, step):
                return True
        return False

    can(0, 0)
    return len(seen)


print(" n đá | số lời gọi | số trạng thái khác nhau")
for m in range(8, 23, 4):
    stones = list(range(m)) + [2 * m - 2]
    _, calls = bruteforce_calls(stones)
    print(f"{len(stones):>5} | {calls:>10} | {distinct_states(stones):>8}")

 n đá | số lời gọi | số trạng thái khác nhau
    9 |         36 |       15
   13 |        279 |       29
   17 |       2165 |       46
   21 |      16795 |       66


### 5.1 Có bao nhiêu trạng thái? Bổ đề "bước nhảy thứ `t` không vượt quá `t`"

Số lời gọi tăng hàm mũ, nhưng số trạng thái khác nhau tăng rất chậm. Vì sao? Ta chứng minh một bổ đề đơn giản mà rất mạnh (sẽ dùng lại ở mẹo cắt tỉa).

**Bổ đề.** Bước nhảy **thứ `t`** của ếch (đếm từ 1) có độ dài **`≤ t`**.

*Chứng minh bằng lời:* bước thứ 1 dài đúng `1`. Mỗi bước sau dài hơn bước trước **nhiều nhất 1** (`k + 1`). Vậy bước 2 `≤ 2`, bước 3 `≤ 3`, ..., bước `t` `≤ t`. ∎

**Hệ quả.** Khi ếch đứng ở **đá chỉ số `i`**, bước cuối `k ≤ i`.

*Vì sao:* mỗi lần nhảy ếch tiến tới một hòn đá có chỉ số **lớn hơn** (nhảy về phía trước, đá tăng ngặt), nên để tới đá chỉ số `i` ếch nhảy **nhiều nhất `i` lần**. Bước cuối là bước thứ `t ≤ i`, nên `k ≤ t ≤ i`.

Ví dụ số: ở Example 1, đá chỉ số `6` (vị trí `12`) chỉ có thể được tới với `k ≤ 6`; thực tế `k = 4`.

**Kết luận về số trạng thái:** có `n` hòn đá, mỗi đá có tối đa `n` giá trị `k` → **tối đa `n²` trạng thái** (`2000² = 4 000 000`, chỉ là cận trên; thực tế ít hơn nhiều). Nếu mỗi trạng thái chỉ được **tính một lần**, mỗi lần làm `O(1)` việc (thử 3 bước, tra `set`), tổng thời gian là `O(n²)`. Đó là mục tiêu của memoization.

Cell dưới kiểm chứng hệ quả `k ≤ i` trên Example 1 và trên 300 bộ đá ngẫu nhiên (dùng một hàm nhỏ liệt kê **mọi** `k` có thể tới mỗi đá).

In [14]:
def reachable_ks(stones):
    # Liệt kê mọi k có thể tới từng đá (lan tỏa từ trái sang phải; mục 7 sẽ giảng kỹ).
    ks = {s: set() for s in stones}
    ks[0].add(0)
    for s in stones:
        for k in ks[s]:
            for step in next_steps(k):
                if s + step in ks:
                    ks[s + step].add(step)
    return ks


ks = reachable_ks(EX1)
for i, s in enumerate(EX1):
    print(f"đá chỉ số {i} (vị trí {s:>2}): k có thể = {sorted(ks[s])}  (cận trên i = {i})")

random.seed(403)
for _ in range(300):
    pos, stones = 0, [0]
    for _ in range(random.randint(1, 30)):
        pos += random.randint(1, 4)
        stones.append(pos)
    ks = reachable_ks(stones)
    assert all(k <= i for i, s in enumerate(stones) for k in ks[s])
print("300 bộ ngẫu nhiên: luôn có k <= i  ✔")

đá chỉ số 0 (vị trí  0): k có thể = [0]  (cận trên i = 0)
đá chỉ số 1 (vị trí  1): k có thể = [1]  (cận trên i = 1)
đá chỉ số 2 (vị trí  3): k có thể = [2]  (cận trên i = 2)
đá chỉ số 3 (vị trí  5): k có thể = [2]  (cận trên i = 3)
đá chỉ số 4 (vị trí  6): k có thể = [1, 3]  (cận trên i = 4)
đá chỉ số 5 (vị trí  8): k có thể = [2, 3]  (cận trên i = 5)
đá chỉ số 6 (vị trí 12): k có thể = [4]  (cận trên i = 6)
đá chỉ số 7 (vị trí 17): k có thể = [5]  (cận trên i = 7)
300 bộ ngẫu nhiên: luôn có k <= i  ✔


## 6. Cách 2 — Đệ quy + memo theo `(vị trí, k)` (cách chính, top-down)

Cách cũ vướng ở đâu? **Tính lại** cùng một trạng thái nhiều lần.
Mấu chốt cải tiến: **ghi sổ** kết quả của từng trạng thái `(pos, k)`.

So với brute force, chỉ thêm 3 dòng:

```python
if (pos, k) in memo:          # 1. đã tính rồi -> đọc sổ
    return memo[(pos, k)]
...
memo[(pos, k)] = result       # 2. tính xong -> ghi sổ
return result                 # 3. trả kết quả
```

Gọi là **top-down** (từ trên xuống) vì ta bắt đầu từ câu hỏi lớn `can(0, 0)` rồi đệ quy xuống các câu hỏi nhỏ hơn, bài con nào cần thì mới tính.

Lưu ý: khóa của sổ phải là **cả cặp `(pos, k)`**, không phải chỉ `pos`. Mục 2 đã cho thấy `(8, 2)` và `(8, 3)` có đáp án khác nhau; mục 12 sẽ cho thấy code memo chỉ theo `pos` trả sai.

In [15]:
def can_cross_memo(stones: List[int]) -> bool:
    stone_set = set(stones)
    last = stones[-1]
    memo = {}                                  # (pos, k) -> True / False

    def can(pos, k):
        if (pos, k) in memo:                   # đọc sổ
            return memo[(pos, k)]
        if pos == last:
            return True
        result = False
        for step in (k - 1, k, k + 1):
            if step > 0 and pos + step in stone_set and can(pos + step, step):
                result = True
                break
        memo[(pos, k)] = result                # ghi sổ
        return result

    return can(0, 0)


print("EX1:", can_cross_memo(EX1))
print("EX2:", can_cross_memo(EX2))

slow = list(range(22)) + [42]                  # bộ đá làm brute force tốn ~47 000 lời gọi
t0 = time.perf_counter(); r1 = can_cross_bruteforce(slow); t1 = time.perf_counter()
r2 = can_cross_memo(slow); t2 = time.perf_counter()
print(f"bộ 23 đá: brute = {r1} ({t1 - t0:.4f}s) | memo = {r2} ({t2 - t1:.5f}s)")

EX1: True
EX2: False
bộ 23 đá: brute = False (0.0084s) | memo = False (0.00007s)


### 6.1 Viết gọn bằng `@lru_cache`

Thay vì tự quản lý `memo`, đặt `@lru_cache(maxsize=None)` lên hàm `can`. Hai tham số `pos, k` đều là số nguyên nên hash được, và Python tự ghi sổ theo đúng cặp `(pos, k)`.

Hàm `can` được định nghĩa **bên trong** `can_cross_lru`, nên mỗi lần gọi `can_cross_lru` là một cuốn sổ **mới**. Nếu đặt `can` ở ngoài (cấp module), sổ của bộ đá trước sẽ bị dùng nhầm cho bộ đá sau, vì sổ chỉ nhớ `(pos, k)` chứ không nhớ `stones`.

In [16]:
def can_cross_lru(stones: List[int]) -> bool:
    stone_set = set(stones)
    last = stones[-1]

    @lru_cache(maxsize=None)
    def can(pos, k):
        if pos == last:
            return True
        for step in (k - 1, k, k + 1):
            if step > 0 and pos + step in stone_set and can(pos + step, step):
                return True
        return False

    answer = can(0, 0)
    print("   cache_info:", can.cache_info())
    return answer


print("EX1:", can_cross_lru(EX1))
print("bộ 23 đá:", can_cross_lru(list(range(22)) + [42]))

   cache_info: CacheInfo(hits=0, misses=9, maxsize=None, currsize=9)
EX1: True
   cache_info: CacheInfo(hits=84, misses=77, maxsize=None, currsize=77)
bộ 23 đá: False


### 6.2 Trace memo: thấy tận mắt "trúng sổ"

Dùng bộ đá nhỏ `[0, 1, 2, 3, 4, 5, 9]` (đáp án `False`, nên phải thử hết, dễ thấy trúng sổ). Hàm `trace_memo` in `→` khi **vào** một trạng thái, `ghi sổ` khi tính xong, và `HIT` khi đọc lại từ sổ.

Điểm cần quan sát:

- `(5, 1)` được tính lần đầu theo đường `0→1→2→3→4→5` và ghi sổ `False`. Về sau, trạng thái `(4, 2)` (đường `0→1→2→4`) cũng muốn nhảy bước 1 tới `(5, 1)`: lần này **HIT**, đọc sổ luôn, không tính lại cả cây con.
- Tương tự, `(3, 2)` (đường `0→1→3`) gặp lại `(4, 1)` và `(5, 2)` đã có trong sổ: thêm 2 lần **HIT**.
- Tổng số lời gọi rất ít so với brute force, và **số mục trong sổ = số trạng thái khác nhau**.

In [17]:
def trace_memo(stones):
    stone_set = set(stones)
    last = stones[-1]
    memo = {}
    stats = {"calls": 0, "hits": 0}

    def can(pos, k, depth):
        ind = "  " * depth
        stats["calls"] += 1
        if (pos, k) in memo:
            stats["hits"] += 1
            print(f"{ind}({pos},{k}) HIT sổ -> {memo[(pos, k)]}")
            return memo[(pos, k)]
        if pos == last:
            print(f"{ind}({pos},{k}) là đá cuối -> True")
            return True
        print(f"{ind}→ ({pos},{k})")
        result = False
        for step in next_steps(k):
            if pos + step in stone_set and can(pos + step, step, depth + 1):
                result = True
                break
        memo[(pos, k)] = result
        print(f"{ind}  ghi sổ ({pos},{k}) = {result}")
        return result

    answer = can(0, 0, 0)
    print(f"Kết quả = {answer} | lời gọi = {stats['calls']} | HIT = {stats['hits']} | mục trong sổ = {len(memo)}")
    return answer

trace_memo([0, 1, 2, 3, 4, 5, 9]);

→ (0,0)
  → (1,1)
    → (2,1)
      → (3,1)
        → (4,1)
          → (5,1)
            ghi sổ (5,1) = False
          ghi sổ (4,1) = False
        → (5,2)
          ghi sổ (5,2) = False
        ghi sổ (3,1) = False
      → (4,2)
        (5,1) HIT sổ -> False
        ghi sổ (4,2) = False
      ghi sổ (2,1) = False
    → (3,2)
      (4,1) HIT sổ -> False
      (5,2) HIT sổ -> False
      ghi sổ (3,2) = False
    ghi sổ (1,1) = False
  ghi sổ (0,0) = False
Kết quả = False | lời gọi = 12 | HIT = 3 | mục trong sổ = 9


### 6.3 Độ phức tạp của cách memo

- **Số trạng thái:** tối đa `n` đá × `n` giá trị `k` (bổ đề mục 5.1) = `O(n²)`.
- **Công việc mỗi trạng thái** (không tính lời gọi con): thử 3 bước, mỗi bước một phép tra `set` trung bình `O(1)` → `O(1)`.
- **Time:** `O(n²)` (trung bình, do dùng bảng băm).
- **Space:** `O(n²)` cho sổ memo + `O(n)` cho `stone_set` + `O(n)` cho độ sâu đệ quy (chuỗi nhảy dài nhất đi qua tối đa `n` đá).

Với `n = 2000`, `n² = 4 000 000`, hoàn toàn trong khả năng. Điểm yếu duy nhất: **đệ quy sâu tới ~2000 tầng**, bắt buộc phải `sys.setrecursionlimit` (đã làm ở mục 3.6). Hai cách tiếp theo tránh hẳn đệ quy.

## 7. Cách 3 — DP bottom-up: "lan tỏa" tập bước nhảy

### 7.1 Đổi góc nhìn: thay vì *hỏi*, ta *đẩy*

Top-down hỏi: *"từ `(pos, k)` có tới đích không?"* rồi đệ quy.
Bottom-up đi ngược lại: duyệt các hòn đá **từ trái sang phải**, và với mỗi đá ghi lại **tất cả các bước nhảy `k` mà ếch có thể dùng để đáp xuống nó**:

```text
dp[s] = tập các k sao cho ếch có thể đáp xuống đá s bằng một bước dài k
```

Cấu trúc dữ liệu: một `dict` **đá → `set`**. Khởi tạo mỗi đá một tập rỗng; riêng `dp[0] = {0}` (mẹo `k = 0`).

Quy tắc **lan tỏa** (push): khi xử lý đá `s`, với mỗi `k` trong `dp[s]`, với mỗi `step` trong `k-1, k, k+1` (`> 0`): nếu `s + step` là đá thì **thêm `step` vào `dp[s + step]`**.

Cuối cùng: đáp án là `True` khi và chỉ khi `dp[last]` **khác rỗng**.

Cell dưới làm quen với "dict của các set" trước.

In [18]:
mini = [0, 1, 3]
dp = {s: set() for s in mini}      # dict comprehension: mỗi đá một set rỗng RIÊNG
print("khởi tạo        :", dp)
dp[0].add(0)
print("dp[0].add(0)    :", dp)
dp[1].add(1)                       # từ 0 nhảy 1 tới 1
dp[3].add(2)                       # từ 1 nhảy 2 tới 3
print("sau 2 lần đẩy   :", dp)
print("dp[3] khác rỗng?:", bool(dp[3]), "| bool(set()) =", bool(set()))
print("4 có phải đá?   :", 4 in dp)  # 'in' trên dict kiểm tra KEY -> dùng luôn làm 'tập vị trí đá'

khởi tạo        : {0: set(), 1: set(), 3: set()}
dp[0].add(0)    : {0: {0}, 1: set(), 3: set()}
sau 2 lần đẩy   : {0: {0}, 1: {1}, 3: {2}}
dp[3] khác rỗng?: True | bool(set()) = False
4 có phải đá?   : False


### 7.2 Vì sao duyệt đá từ trái sang phải là đủ?

Ếch chỉ nhảy **về phía trước**, nên mọi bước nhảy đáp xuống đá `s` đều xuất phát từ một đá **nhỏ hơn** `s`. Khi vòng lặp tới `s`, mọi đá nhỏ hơn đã được xử lý xong và đã "đẩy" hết vào `dp[s]`. Do đó **`dp[s]` đã đầy đủ** đúng lúc ta cần dùng nó. Đây là **thứ tự tính toán** (topological order) tự nhiên của bài.

Thêm một chi tiết kỹ thuật: khi đang duyệt `for k in dp[s]`, ta chỉ thêm vào `dp[s + step]` với `step > 0`, tức **một set khác** `dp[s]`. Nhờ vậy không bị lỗi "set thay đổi kích thước trong khi duyệt" (mục 12 sẽ cho xem lỗi này khi quên `step > 0`).

Dùng chính `dp` (key là các vị trí đá) để kiểm tra `s + step in dp`, khỏi cần thêm `stone_set`.

In [19]:
def can_cross_dp(stones: List[int]) -> bool:
    dp = {s: set() for s in stones}      # đá -> tập các bước k đã đáp xuống được
    dp[0].add(0)                         # mẹo: "vừa nhảy bước 0" để tới đá 0
    for s in stones:                     # trái -> phải
        for k in dp[s]:
            for step in (k - 1, k, k + 1):
                if step > 0 and s + step in dp:
                    dp[s + step].add(step)       # lan tỏa sang đá phía trước
    return len(dp[stones[-1]]) > 0


print("EX1:", can_cross_dp(EX1))
print("EX2:", can_cross_dp(EX2))
print("[0, 1]:", can_cross_dp([0, 1]), "| [0, 2]:", can_cross_dp([0, 2]))

EX1: True
EX2: False
[0, 1]: True | [0, 2]: False


### 7.3 Trace bảng DP cho Example 1 `[0,1,3,5,6,8,12,17]`

Bảng dưới là trạng thái **tại lúc xử lý** mỗi đá (lúc đó `dp[s]` đã đầy đủ). Cột cuối là các lần "đẩy" thành công; vị trí nước thì bỏ qua.

| Đá `s` | `dp[s]` | Các bước thử | Đẩy thành công |
|---:|---|---|---|
| 0 | `{0}` | k=0: `1` | `0+1→1` (thêm 1 vào `dp[1]`) |
| 1 | `{1}` | k=1: `1, 2` | `2` là nước; `1+2→3` |
| 3 | `{2}` | k=2: `1, 2, 3` | `4` nước; `3+2→5`; `3+3→6` |
| 5 | `{2}` | k=2: `1, 2, 3` | `5+1→6`; `7` nước; `5+3→8` |
| 6 | `{1, 3}` | k=1: `1, 2`; k=3: `2, 3, 4` | k=1: `6+2→8`; k=3: `6+2→8` (trùng, set giữ 1 bản); `9, 10` nước |
| 8 | `{2, 3}` | k=2: `1, 2, 3`; k=3: `2, 3, 4` | k=2: `9, 10, 11` nước (kẹt); k=3: `8+4→12` |
| 12 | `{4}` | k=4: `3, 4, 5` | `15, 16` nước; `12+5→17` |
| 17 | `{5}` | đá cuối | `dp[17] = {5}` khác rỗng → **True** |

Đọc bảng:

- Đá `6` được tới bằng **hai** bước khác nhau (`k = 3` từ đá 3, `k = 1` từ đá 5) → `dp[6] = {1, 3}`.
- Đá `8` có `dp[8] = {2, 3}`: đúng hai trạng thái `(8, 2)` (kẹt) và `(8, 3)` (đi tiếp) đã phân tích ở mục 2.
- `dp[17] = {5}`: ếch đáp xuống đá cuối bằng bước 5, khớp lời giải thích của đề.

Hàm `trace_dp` in đúng bảng này (duyệt `k` theo thứ tự tăng cho dễ đọc):

In [20]:
def trace_dp(stones):
    dp = {s: set() for s in stones}
    dp[0].add(0)
    for s in stones:
        pushes = []
        for k in sorted(dp[s]):
            for step in next_steps(k):
                if s + step in dp:
                    pushes.append(f"{s}+{step}→{s + step}(k={k})")
                    dp[s + step].add(step)
        print(f"đá {s:>2} | dp = {str(sorted(dp[s])):<7} | đẩy: {', '.join(pushes) if pushes else '—'}")
    answer = len(dp[stones[-1]]) > 0
    print("=> dp[last] =", sorted(dp[stones[-1]]), "->", answer)
    return answer

trace_dp(EX1);

đá  0 | dp = [0]     | đẩy: 0+1→1(k=0)
đá  1 | dp = [1]     | đẩy: 1+2→3(k=1)
đá  3 | dp = [2]     | đẩy: 3+2→5(k=2), 3+3→6(k=2)
đá  5 | dp = [2]     | đẩy: 5+1→6(k=2), 5+3→8(k=2)
đá  6 | dp = [1, 3]  | đẩy: 6+2→8(k=1), 6+2→8(k=3)
đá  8 | dp = [2, 3]  | đẩy: 8+4→12(k=3)
đá 12 | dp = [4]     | đẩy: 12+5→17(k=4)
đá 17 | dp = [5]     | đẩy: —
=> dp[last] = [5] -> True


### 7.4 Trace Example 2 `[0,1,2,3,4,8,9,11]`: thấy rõ chỗ đứt

Dự đoán trước khi chạy: trên đoạn `0..4`, bước nhảy chỉ tăng chậm. Ở đá `4`, `dp[4] = {1, 2}`, nên bước tiếp theo dài nhất là `2 + 1 = 3`, đáp xuống vị trí `7`, là nước. Không ai đẩy được vào `dp[8]`, và vì thế `dp[9]`, `dp[11]` cũng rỗng.

| Đá `s` | `dp[s]` | Đẩy thành công |
|---:|---|---|
| 0 | `{0}` | `0+1→1` |
| 1 | `{1}` | `1+1→2`; `1+2→3` |
| 2 | `{1}` | `2+1→3`; `2+2→4` |
| 3 | `{1, 2}` | k=1: `3+1→4`; k=2: `3+1→4` (trùng) |
| 4 | `{1, 2}` | không có (5, 6, 7 đều là nước) |
| 8, 9, 11 | `{}` | rỗng → **False** |

In [21]:
trace_dp(EX2);

đá  0 | dp = [0]     | đẩy: 0+1→1(k=0)
đá  1 | dp = [1]     | đẩy: 1+1→2(k=1), 1+2→3(k=1)
đá  2 | dp = [1]     | đẩy: 2+1→3(k=1), 2+2→4(k=1)
đá  3 | dp = [1, 2]  | đẩy: 3+1→4(k=1), 3+1→4(k=2)
đá  4 | dp = [1, 2]  | đẩy: —
đá  8 | dp = []      | đẩy: —
đá  9 | dp = []      | đẩy: —
đá 11 | dp = []      | đẩy: —
=> dp[last] = [] -> False


### 7.5 Độ phức tạp của DP bottom-up

- Mỗi cặp `(s, k)` với `k ∈ dp[s]` là một trạng thái, tối đa `O(n²)` cặp (bổ đề mục 5.1). Mỗi cặp làm 3 phép thử `O(1)`.
- **Time:** `O(n²)` (trung bình, do tra bảng băm).
- **Space:** `O(n²)` cho tổng kích thước các `set`.
- **Không đệ quy** → không lo `RecursionError`.

Khác biệt với top-down: bottom-up **luôn tính hết** mọi trạng thái tới được, kể cả khi đã sớm có đường tới đích. Top-down thì dừng ngay khi tìm thấy một đường (`return True`). Ngược lại, bottom-up không tốn chi phí gọi hàm. Ở mục 9 ta sẽ đo thử.

## 8. Cách 4 — BFS trên trạng thái `(đá, k)`

### 8.1 Hàng đợi `deque` và BFS từ con số 0

**BFS** (Breadth-First Search, duyệt theo chiều rộng) là cách đi qua một "đồ thị" theo **từng lớp**: trước hết các đỉnh cách xuất phát 1 bước, rồi 2 bước, rồi 3 bước...

Công cụ: **hàng đợi** (queue), vào trước ra trước (FIFO). Trong Python dùng `collections.deque`:

- `q.append(x)`: thêm vào **cuối** hàng.
- `q.popleft()`: lấy ra ở **đầu** hàng, `O(1)` (khác `list.pop(0)` tốn `O(n)`).

Ở bài này, mỗi **đỉnh** là một trạng thái `(pos, k)`, mỗi **cạnh** là một bước nhảy hợp lệ. Ta cần thêm một `set` tên `visited` để không đưa một trạng thái vào hàng đợi hai lần (giống vai trò của sổ memo).

In [22]:
q = deque()
q.append("A")
q.append("B")
q.append("C")
print("hàng đợi       :", q)
print("popleft        :", q.popleft())     # A ra trước (vào trước)
q.append("D")
print("sau khi thêm D :", q)
while q:                                   # deque rỗng thì bool(q) == False
    print("  xử lý", q.popleft())

hàng đợi       : deque(['A', 'B', 'C'])
popleft        : A
sau khi thêm D : deque(['B', 'C', 'D'])
  xử lý B
  xử lý C
  xử lý D


### 8.2 Code BFS

```text
q = [(0, 0)], visited = {(0, 0)}
lặp khi q còn phần tử:
    lấy (pos, k) ở đầu hàng
    nếu pos == last: trả True
    với mỗi step trong k-1, k, k+1 (> 0):
        nếu pos+step là đá và (pos+step, step) chưa visited: đánh dấu, đưa vào hàng
hết hàng mà chưa tới đích: trả False
```

Mỗi trạng thái vào hàng đợi đúng một lần → cùng độ phức tạp `O(n²)` như memo, nhưng **không đệ quy**. Phần thưởng thêm: BFS duyệt theo lớp, nên **lần đầu** chạm đá cuối cũng là lúc dùng **ít lần nhảy nhất** (bài tập 6 sẽ khai thác).

In [23]:
def can_cross_bfs(stones: List[int]) -> bool:
    stone_set = set(stones)
    last = stones[-1]
    q = deque([(0, 0)])
    visited = {(0, 0)}
    while q:
        pos, k = q.popleft()
        if pos == last:
            return True
        for step in (k - 1, k, k + 1):
            nxt = pos + step
            if step > 0 and nxt in stone_set and (nxt, step) not in visited:
                visited.add((nxt, step))
                q.append((nxt, step))
    return False


print("EX1:", can_cross_bfs(EX1))
print("EX2:", can_cross_bfs(EX2))

EX1: True
EX2: False


### 8.3 Trace BFS theo từng lớp trên Example 1

Lớp `d` = các trạng thái tới được bằng đúng `d` lần nhảy (lần đầu tiên). Dự đoán:

| Lớp (số lần nhảy) | Trạng thái `(pos, k)` |
|---:|---|
| 0 | `(0,0)` |
| 1 | `(1,1)` |
| 2 | `(3,2)` |
| 3 | `(5,2)`, `(6,3)` |
| 4 | `(6,1)`, `(8,3)`, `(8,2)` |
| 5 | `(12,4)` |
| 6 | `(17,5)` → tới đích |

Khi mở rộng lớp 4 để tạo lớp 5: `(6,1)` muốn đẩy `(8,2)` nhưng `(8,2)` đã có trong `visited` (do `(6,3)` đẩy vào từ trước) nên bị bỏ qua; `(8,2)` thì kẹt (`9, 10, 11` là nước). Chỉ `(8,3)` sinh ra `(12,4)`.

Với Example 2, sau lớp 3 không còn trạng thái mới nào (từ đá `4` mọi bước đều rơi xuống nước), hàng đợi cạn → `False`.

In [24]:
def trace_bfs(stones):
    stone_set = set(stones)
    last = stones[-1]
    layer = [(0, 0)]
    visited = {(0, 0)}
    d = 0
    while layer:
        print(f"lớp {d}: {layer}")
        if any(pos == last for pos, _ in layer):
            print(f"=> chạm đá cuối sau {d} lần nhảy: True")
            return True
        nxt_layer = []
        for pos, k in layer:
            for step in next_steps(k):
                nxt = pos + step
                if nxt in stone_set and (nxt, step) not in visited:
                    visited.add((nxt, step))
                    nxt_layer.append((nxt, step))
        layer = nxt_layer
        d += 1
    print("=> hết trạng thái, không tới được: False")
    return False

trace_bfs(EX1)
print()
trace_bfs(EX2);

lớp 0: [(0, 0)]
lớp 1: [(1, 1)]
lớp 2: [(3, 2)]
lớp 3: [(5, 2), (6, 3)]
lớp 4: [(6, 1), (8, 3), (8, 2)]
lớp 5: [(12, 4)]
lớp 6: [(17, 5)]
=> chạm đá cuối sau 6 lần nhảy: True

lớp 0: [(0, 0)]
lớp 1: [(1, 1)]
lớp 2: [(2, 1), (3, 2)]
lớp 3: [(3, 1), (4, 2), (4, 1)]
=> hết trạng thái, không tới được: False


## 9. Cách 5 — Cắt tỉa sớm: nếu `stones[i] - stones[i-1] > i` thì `False`

### 9.1 Phát biểu và chứng minh

Dùng lại **bổ đề** ở mục 5.1: *bước nhảy thứ `t` dài `≤ t`*, và hệ quả: *đứng ở đá chỉ số `j` thì đã nhảy nhiều nhất `j` lần*.

**Mệnh đề.** Nếu tồn tại `i ≥ 1` mà `stones[i] - stones[i-1] > i` thì ếch **không thể** qua sông.

*Chứng minh bằng lời:* muốn tới đá cuối, ếch phải có **một lần nhảy "vượt khe"**: nhảy từ một đá chỉ số `j ≤ i-1` tới một đá chỉ số `≥ i`.

- Lúc đứng ở đá `j`, ếch đã nhảy `≤ j` lần, nên lần nhảy vượt khe là lần thứ `t ≤ j + 1 ≤ i`, tức độ dài `≤ i`.
- Nhưng lần nhảy đó đi từ vị trí `≤ stones[i-1]` tới vị trí `≥ stones[i]`, nên độ dài `≥ stones[i] - stones[i-1] > i`.

Mâu thuẫn. ∎

Ví dụ số:

- `[0, 2]`: `i = 1`, khe dài `2 > 1` → `False` (đúng: bước đầu phải là 1).
- `[0, 1, 3, 6, 10, 15, 21]`: các khe `1, 2, 3, 4, 5, 6` đúng bằng `i` → **qua được bước kiểm tra**, và thực tế ếch nhảy `1, 2, 3, 4, 5, 6`, tới đích.
- `[0, 1, 3, 6, 10, 15, 22]`: khe cuối `7 > 6` → `False` ngay, không cần DP.

**Lưu ý quan trọng:** đây là điều kiện **cần**, không phải **đủ**. Example 2 có các khe `1,1,1,1,4,1,2` với `i = 1..7`, khe `4` ở `i = 5` không vượt `5`, nên **qua được** bước kiểm tra, nhưng đáp án vẫn là `False`. Vì vậy cắt tỉa chỉ là **bước lọc nhanh `O(n)`** đặt trước một thuật toán đầy đủ (memo/DP/BFS), không thay thế chúng.

In [25]:
def first_bad_gap(stones):
    # Trả chỉ số i đầu tiên có stones[i] - stones[i-1] > i, hoặc None.
    for i in range(1, len(stones)):
        if stones[i] - stones[i - 1] > i:
            return i
    return None


def can_cross_pruned(stones: List[int]) -> bool:
    if first_bad_gap(stones) is not None:      # lọc nhanh O(n)
        return False
    return can_cross_dp(stones)                # vẫn cần thuật toán đầy đủ


for stones in ([0, 2], [0, 1, 3, 6, 10, 15, 21], [0, 1, 3, 6, 10, 15, 22], EX1, EX2):
    bad = first_bad_gap(stones)
    print(f"{str(stones):<28} khe xấu tại i = {str(bad):<4} | pruned = {can_cross_pruned(stones)}")

[0, 2]                       khe xấu tại i = 1    | pruned = False
[0, 1, 3, 6, 10, 15, 21]     khe xấu tại i = None | pruned = True
[0, 1, 3, 6, 10, 15, 22]     khe xấu tại i = 6    | pruned = False
[0, 1, 3, 5, 6, 8, 12, 17]   khe xấu tại i = None | pruned = True
[0, 1, 2, 3, 4, 8, 9, 11]    khe xấu tại i = None | pruned = False


### 9.2 Đo thời gian các cách trên input lớn (`n = 2000`)

Ba bộ đá "khó" với kích thước tối đa của đề:

1. **Tam giác** `0, 1, 3, 6, 10, ...` (vị trí `t(t+1)/2`): chỉ có **một** đường, nhảy `1, 2, 3, ..., 1999`, đệ quy sâu ~2000 tầng. Đáp án `True`.
2. **Liền kề + đá xa** `0..1998` rồi `10^9`: khe cuối khổng lồ, đáp án `False`. Không cắt tỉa thì phải duyệt mọi trạng thái; có cắt tỉa thì xong trong `O(n)`.
3. **Liền kề + đá vừa xa** `0..1998` rồi `1998 + 1999`: khe cuối `1999 ≤ 1999` nên cắt tỉa **không** giúp, đáp án `False`, phải duyệt hết trạng thái.

(Không đưa brute force vào: nó sẽ không bao giờ chạy xong.)

In [26]:
triangle = [t * (t + 1) // 2 for t in range(2000)]
far_last = list(range(1999)) + [10**9]
near_last = list(range(1999)) + [1998 + 1999]

methods = [("memo dict", can_cross_memo), ("dp bottom-up", can_cross_dp),
           ("bfs", can_cross_bfs), ("pruned + dp", can_cross_pruned)]

for name_input, stones in [("tam giác", triangle), ("liền kề + đá xa", far_last), ("liền kề + vừa xa", near_last)]:
    row = []
    for name, f in methods:
        t0 = time.perf_counter()
        r = f(stones)
        row.append(f"{name}={r} {time.perf_counter() - t0:.3f}s")
    print(f"{name_input:<17}| " + " | ".join(row))
print("Số trạng thái tới được của 'liền kề + vừa xa':",
      sum(len(v) for v in reachable_ks(near_last).values()))

tam giác         | memo dict=True 0.001s | dp bottom-up=True 0.001s | bfs=True 0.001s | pruned + dp=True 0.001s


liền kề + đá xa  | memo dict=False 0.075s | dp bottom-up=False 0.028s | bfs=False 0.086s | pruned + dp=False 0.000s


liền kề + vừa xa | memo dict=False 0.062s | dp bottom-up=False 0.023s | bfs=False 0.047s | pruned + dp=False 0.027s


Số trạng thái tới được của 'liền kề + vừa xa': 82275


Nhận xét:

- Trên bộ **tam giác**, cả bốn cách đều nhanh vì chỉ có một đường; nhưng bản memo đệ quy chỉ chạy được vì ta đã nâng giới hạn đệ quy (mục 3.6). Với giới hạn mặc định 1000 nó sẽ báo `RecursionError` (mục 12, lỗi 6).
- Trên bộ **liền kề + đá xa**, cắt tỉa trả lời gần như tức thì; các cách khác phải duyệt khoảng 80 000 trạng thái.
- Trên bộ **liền kề + vừa xa**, cắt tỉa **không** cứu được: mọi cách đều phải duyệt hết trạng thái. Số trạng thái thực tế (~80 000) nhỏ hơn rất nhiều so với cận trên `n² = 4 000 000`, vì để có bước dài `k` thì trước đó phải đi được ít nhất `1 + 2 + ... + k` đơn vị đường.

## 10. Code nộp LeetCode

Bản chính: **top-down memo** (cách 2) + **cắt tỉa** (cách 5) + **nâng giới hạn đệ quy**. Những điểm chuyển từ hàm demo sang `class Solution`:

- Đúng chữ ký `def canCross(self, stones: List[int]) -> bool` (LeetCode đã `from typing import List` sẵn; trong notebook ta đã import ở đầu).
- Hàm `can` có `@lru_cache` đặt **bên trong** method, nên mỗi test của LeetCode dùng sổ mới.
- `sys.setrecursionlimit(10000)` vì chuỗi đệ quy có thể sâu ~2000 tầng.
- Thử bước `k + 1` trước rồi tới `k`, `k - 1`: thứ tự thử **không ảnh hưởng tính đúng** (vẫn thử đủ 3 bước), chỉ thường giúp chạm đích sớm hơn khi đáp án là `True`, vì bước dài đi xa nhanh hơn.

In [27]:
class Solution:
    def canCross(self, stones: List[int]) -> bool:
        # Cắt tỉa O(n): bước nhảy tới/vượt đá i không thể dài hơn i.
        for i in range(1, len(stones)):
            if stones[i] - stones[i - 1] > i:
                return False

        sys.setrecursionlimit(10000)          # đệ quy có thể sâu ~len(stones)
        stone_set = set(stones)
        last = stones[-1]

        @lru_cache(maxsize=None)
        def can(pos: int, k: int) -> bool:
            if pos == last:
                return True
            for step in (k + 1, k, k - 1):
                if step > 0 and pos + step in stone_set and can(pos + step, step):
                    return True
            return False

        return can(0, 0)

In [28]:
sol = Solution()
tests = [
    (EX1, True),                               # Example 1
    (EX2, False),                              # Example 2
    ([0, 1], True),                            # nhỏ nhất, nhảy 1 là tới
    ([0, 2], False),                           # bước đầu phải là 1
    ([0, 1, 3], True),                         # 1 rồi 2
    ([0, 1, 4], False),                        # sau bước 1 chỉ được 0/1/2
    ([0, 1, 2, 3, 4, 5, 6], True),             # đá liền kề
    ([0, 1, 3, 6, 10, 15, 21], True),          # khe đúng bằng i
    ([0, 1, 3, 6, 10, 15, 22], False),         # khe cuối 7 > 6: cắt tỉa
    ([0, 1, 3, 6, 7], False),                  # tới 6 với k=3 thì không xuống được 7
    ([0, 1, 2, 3, 6, 9], True),                # bẫy memo chỉ theo vị trí
    ([0, 2147483647], False),                  # vị trí cực lớn
    (triangle, True),                          # n = 2000, đệ quy sâu
    (near_last, False),                        # n = 2000, phải duyệt hết
]
for stones, expected in tests:
    got = sol.canCross(stones)
    label = str(stones) if len(stones) <= 10 else f"<{len(stones)} đá>"
    print(f"{label:<32} -> {got!s:<5} (mong đợi {expected})")
    assert got == expected
print("Tất cả test PASS!")

[0, 1, 3, 5, 6, 8, 12, 17]       -> True  (mong đợi True)
[0, 1, 2, 3, 4, 8, 9, 11]        -> False (mong đợi False)
[0, 1]                           -> True  (mong đợi True)
[0, 2]                           -> False (mong đợi False)
[0, 1, 3]                        -> True  (mong đợi True)
[0, 1, 4]                        -> False (mong đợi False)
[0, 1, 2, 3, 4, 5, 6]            -> True  (mong đợi True)
[0, 1, 3, 6, 10, 15, 21]         -> True  (mong đợi True)
[0, 1, 3, 6, 10, 15, 22]         -> False (mong đợi False)
[0, 1, 3, 6, 7]                  -> False (mong đợi False)
[0, 1, 2, 3, 6, 9]               -> True  (mong đợi True)
[0, 2147483647]                  -> False (mong đợi False)
<2000 đá>                        -> True  (mong đợi True)
<2000 đá>                        -> False (mong đợi False)
Tất cả test PASS!


### 10.1 Bản nộp thay thế: DP bottom-up (không đệ quy)

Nếu không muốn đụng tới giới hạn đệ quy, nộp bản bottom-up (cách 3). Trong notebook ta đặt tên `SolutionDP` để không ghi đè `Solution` ở trên; **khi nộp, đổi tên lớp thành `Solution`**.

In [29]:
class SolutionDP:
    def canCross(self, stones: List[int]) -> bool:
        for i in range(1, len(stones)):
            if stones[i] - stones[i - 1] > i:
                return False
        dp = {s: set() for s in stones}
        dp[0].add(0)
        for s in stones:
            for k in dp[s]:
                for step in (k - 1, k, k + 1):
                    if step > 0 and s + step in dp:
                        dp[s + step].add(step)
        return len(dp[stones[-1]]) > 0


sol_dp = SolutionDP()
for stones, expected in tests:
    assert sol_dp.canCross(stones) == expected
print("SolutionDP: tất cả", len(tests), "test PASS!")

SolutionDP: tất cả 14 test PASS!


### 10.2 Kiểm thử chéo tất cả các cách

Sinh ngẫu nhiên nhiều bộ đá nhỏ (để brute force còn chạy được), với `random.seed` cố định để kết quả lặp lại được. Khe giữa hai đá chọn ngẫu nhiên trong `1..3`, nên có cả ca `True` lẫn `False`. Mọi cách phải trả **cùng một đáp án** với brute force (cách chậm nhưng hiển nhiên đúng).

In [30]:
def can_cross_lru_quiet(stones):
    # Bản lru_cache không in cache_info, để dùng trong test chéo.
    stone_set, last = set(stones), stones[-1]

    @lru_cache(maxsize=None)
    def can(pos, k):
        if pos == last:
            return True
        return any(step > 0 and pos + step in stone_set and can(pos + step, step)
                   for step in (k - 1, k, k + 1))

    return can(0, 0)


random.seed(2024)
all_methods = {
    "memo": can_cross_memo, "lru": can_cross_lru_quiet, "dp": can_cross_dp,
    "bfs": can_cross_bfs, "pruned": can_cross_pruned,
    "Solution": Solution().canCross, "SolutionDP": SolutionDP().canCross,
}
count = {True: 0, False: 0}
for _ in range(3000):
    pos, stones = 0, [0]
    for _ in range(random.randint(1, 14)):
        pos += random.randint(1, 3)
        stones.append(pos)
    expected = can_cross_bruteforce(stones)
    count[expected] += 1
    for name, f in all_methods.items():
        assert f(stones) == expected, (name, stones)
print("3000 bộ ngẫu nhiên khớp brute force | số ca True/False:", count)

3000 bộ ngẫu nhiên khớp brute force | số ca True/False: {True: 519, False: 2481}


## 11. Vì sao đúng? Độ phức tạp và so sánh

### 11.1 Tính đúng (invariant bằng lời)

**Top-down memo.** `can(pos, k)` trả `True` **khi và chỉ khi** từ trạng thái `(pos, k)` có một dãy nhảy hợp lệ tới đá cuối.

- Nếu `pos == last`: đúng hiển nhiên.
- Ngược lại, bước tiếp theo **bắt buộc** là một trong `k-1, k, k+1` (dương) và phải đáp xuống đá. Hàm thử **đủ** cả 3 khả năng (không bỏ sót), và chỉ trả `True` khi một khả năng thực sự dẫn tới đích (không trả nhầm).
- Memo đúng vì kết quả **chỉ phụ thuộc `(pos, k)`**: mọi thông tin cần để quyết định tương lai đều nằm trong cặp này, đường đi trong quá khứ không ảnh hưởng.
- Đệ quy luôn dừng vì `pos` tăng ngặt ở mỗi lời gọi con.

**Bottom-up DP.** Bất biến: *sau khi xử lý xong mọi đá `< s`, `dp[s]` chứa đúng tập các bước `k` mà ếch có thể dùng để đáp xuống `s`*.

- Mỗi `k` được thêm vào `dp[s]` đều đến từ một trạng thái tới được `(s', k')` với `k ∈ {k'-1, k', k'+1}` → không thêm nhầm.
- Mọi cách đáp xuống `s` đều xuất phát từ đá `s' < s`, đã được xử lý trước → không bỏ sót.
- Vậy `dp[last]` khác rỗng ⇔ tới được đá cuối.

**BFS.** Đúng là duyệt đồ thị trạng thái từ `(0, 0)`; một đỉnh tới được ⇔ nó được đưa vào hàng đợi. `visited` chỉ chặn đưa **lại** một đỉnh đã có, không chặn đỉnh mới.

**Cắt tỉa.** Đã chứng minh ở mục 9.1: chỉ trả `False` khi chắc chắn không qua được.

### 11.2 Bảng so sánh

| Cách | Ý tưởng | Time | Space | Ghi chú |
|---|---|---|---|---|
| 1. Brute force đệ quy | Thử `k-1, k, k+1`, quay lui | `O(3^n)` (hàm mũ) | `O(n)` độ sâu | Chỉ để hiểu bài; TLE |
| 2. Đệ quy + memo `(pos, k)` | Ghi sổ mỗi trạng thái | `O(n²)` | `O(n²)` | Cách chính; cần `setrecursionlimit` |
| 3. DP bottom-up `dict → set` | Lan tỏa trái → phải | `O(n²)` | `O(n²)` | Không đệ quy; tính hết mọi trạng thái |
| 4. BFS trạng thái | Duyệt theo lớp = số lần nhảy | `O(n²)` | `O(n²)` | Không đệ quy; cho luôn số lần nhảy ít nhất |
| 5. Cắt tỉa khe `> i` | Bổ đề bước thứ `t ≤ t` | `O(n)` (phần lọc) | `O(1)` | Điều kiện cần, ghép trước 2/3/4 |

Ở đây `n = len(stones)`; các `O(n²)` là cận trên số trạng thái `(đá, k)` với `k ≤` chỉ số đá, và là độ phức tạp trung bình do dùng bảng băm (`set`/`dict`).

## 12. Lỗi thường gặp (code sai → vì sao → sửa)

### Lỗi 1: Quên điều kiện `step > 0`

```python
for step in (k - 1, k, k + 1):
    if pos + step in stone_set and can(pos + step, step):   # THIẾU step > 0
```

Từ `(pos, 1)`, bước `0` gọi `can(pos, 0)`; từ `(pos, 0)` bước `0` lại gọi `can(pos, 0)`... **vòng lặp vô hạn** → `RecursionError`. (Bước âm còn tệ hơn: nhảy lùi.)
**Sửa:** luôn kiểm tra `step > 0`.

### Lỗi 2: Sai bước nhảy đầu tiên

```python
return can(0, 1)     # SAI: coi như "vừa nhảy 1" ở đá 0 -> cho phép bước đầu 0, 1, 2
```

Với `[0, 2]`, code này nhảy thẳng `2` và trả `True`, nhưng bước đầu bắt buộc là `1` → đáp án đúng `False`.
**Sửa:** xuất phát từ `can(0, 0)` (chỉ còn bước `1`), hoặc kiểm tra riêng `stones[1] == 1` rồi gọi `can(1, 1)`.

### Lỗi 3: Memo chỉ theo vị trí

```python
if pos in memo: return memo[pos]     # SAI: bỏ mất k
```

Với `[0, 1, 2, 3, 6, 9]`: DFS đi `0→1→2→3` tới `(3, 1)`, kẹt, ghi `memo[3] = False`. Sau đó `(1, 1)` thử bước 2 tới `(3, 2)`, nhưng đọc nhầm `memo[3] = False`, trong khi `(3, 2)` đi tiếp được `3→6→9`. Kết quả sai `False` (đúng là `True`).
**Sửa:** khóa là `(pos, k)`.

Cell dưới chạy cả ba phiên bản sai để thấy tận mắt.

In [31]:
def buggy_no_positive_check(stones):
    stone_set, last = set(stones), stones[-1]
    def can(pos, k):
        if pos == last:
            return True
        for step in (k - 1, k, k + 1):
            if pos + step in stone_set and can(pos + step, step):   # thiếu step > 0
                return True
        return False
    return can(0, 0)


def buggy_first_jump(stones):
    stone_set, last = set(stones), stones[-1]
    @lru_cache(maxsize=None)
    def can(pos, k):
        if pos == last:
            return True
        return any(s > 0 and pos + s in stone_set and can(pos + s, s) for s in (k - 1, k, k + 1))
    return can(0, 1)                                                # sai điểm xuất phát


def buggy_memo_pos_only(stones):
    stone_set, last, memo = set(stones), stones[-1], {}
    def can(pos, k):
        if pos in memo:                                             # khóa thiếu k
            return memo[pos]
        if pos == last:
            return True
        result = any(s > 0 and pos + s in stone_set and can(pos + s, s) for s in (k - 1, k, k + 1))
        memo[pos] = result
        return result
    return can(0, 0)


try:
    buggy_no_positive_check(EX1)
except RecursionError as e:
    print("Lỗi 1 -> RecursionError:", e)
print("Lỗi 2 với [0, 2]            :", buggy_first_jump([0, 2]), "| đúng:", can_cross_memo([0, 2]))
print("Lỗi 3 với [0, 1, 2, 3, 6, 9]:", buggy_memo_pos_only([0, 1, 2, 3, 6, 9]),
      "| đúng:", can_cross_memo([0, 1, 2, 3, 6, 9]))

Lỗi 1 -> RecursionError: maximum recursion depth exceeded
Lỗi 2 với [0, 2]            : True | đúng: False
Lỗi 3 với [0, 1, 2, 3, 6, 9]: False | đúng: True


### Lỗi 4: Bottom-up quên `step > 0` → "set changed size during iteration"

Trong bottom-up, nếu cho phép `step = 0` thì `s + 0 = s` cũng là đá, ta sẽ `dp[s].add(0)` **trong lúc đang duyệt `for k in dp[s]`**. Python cấm thay đổi kích thước một `set` khi đang duyệt nó → `RuntimeError`.
**Sửa:** `step > 0` bảo đảm luôn thêm vào set của đá **khác**.

### Lỗi 5: Tra cứu bằng `list` thay vì `set`

`pos + step in stones` (với `stones` là list) tốn `O(n)` mỗi lần, đẩy tổng thời gian lên `O(n³)`. Với `n = 2000` dễ bị **TLE** (Time Limit Exceeded).
**Sửa:** `stone_set = set(stones)` một lần ở đầu (hoặc dùng luôn `dp` là dict).

### Lỗi 6: Quên nâng giới hạn đệ quy

Bản top-down trên bộ tam giác 2000 đá đệ quy sâu ~2000 tầng, vượt mức mặc định 1000.
**Sửa:** `sys.setrecursionlimit(10000)` hoặc dùng bản bottom-up/BFS.

### Lỗi 7: Trả về `set` thay vì `bool`

`return dp[stones[-1]]` trả về một `set` như `{5}`. Kiểu trả về của đề là `bool`.
**Sửa:** `return len(dp[stones[-1]]) > 0` hoặc `return bool(dp[stones[-1]])`.

Cell dưới minh họa lỗi 4, 5 và 6 (lỗi 6: tạm hạ giới hạn về 1000 rồi khôi phục).

In [32]:
def buggy_dp_step_zero(stones):
    dp = {s: set() for s in stones}
    dp[0].add(0)
    for s in stones:
        for k in dp[s]:
            for step in (k - 1, k, k + 1):
                if s + step in dp:                 # thiếu step > 0
                    dp[s + step].add(step)
    return len(dp[stones[-1]]) > 0

try:
    buggy_dp_step_zero(EX1)
except RuntimeError as e:
    print("Lỗi 4 -> RuntimeError:", e)


def dp_with_list_lookup(stones):
    dp = {s: set() for s in stones}
    dp[0].add(0)
    for s in stones:
        for k in dp[s]:
            for step in (k - 1, k, k + 1):
                if step > 0 and s + step in stones:   # 'in' trên LIST: O(n)
                    dp[s + step].add(step)
    return len(dp[stones[-1]]) > 0

medium = list(range(999)) + [998 + 999]
t0 = time.perf_counter(); dp_with_list_lookup(medium); t1 = time.perf_counter()
can_cross_dp(medium); t2 = time.perf_counter()
print(f"Lỗi 5 -> 1000 đá: tra list {t1 - t0:.3f}s | tra dict/set {t2 - t1:.3f}s")

sys.setrecursionlimit(1000)
try:
    can_cross_memo(triangle)
except RecursionError as e:
    print("Lỗi 6 -> RecursionError với giới hạn 1000:", e)
finally:
    sys.setrecursionlimit(10000)
print("Lỗi 7 -> return set:", {5}, "nhưng cần bool:", bool({5}))

Lỗi 4 -> RuntimeError: Set changed size during iteration


Lỗi 5 -> 1000 đá: tra list 0.273s | tra dict/set 0.010s
Lỗi 6 -> RecursionError với giới hạn 1000: maximum recursion depth exceeded
Lỗi 7 -> return set: {5} nhưng cần bool: True


## 13. Mẹo nhận dạng pattern và bài liên quan

### Dấu hiệu nhận dạng

- **"Bước tiếp theo phụ thuộc bước trước"** (`k-1, k, k+1`, tốc độ, hướng đi, màu vừa dùng...) → trạng thái phải gồm **(vị trí, thông tin về bước trước)**. Chỉ lưu vị trí là sai (lỗi 3).
- **"Có tới được không / có tồn tại cách không"** + số lựa chọn mỗi bước nhỏ → DFS + memo, hoặc BFS/DP trên trạng thái.
- **Tọa độ lớn nhưng số điểm ít** (`2^31` vs `2000`) → làm việc trên **tập điểm** (`set`/`dict`), không tạo mảng theo tọa độ.
- **Đại lượng thay đổi tối đa 1 mỗi bước** → bổ đề kiểu "sau `t` bước thì `≤ t`" giúp **chặn số trạng thái** (tính Big-O) và **cắt tỉa sớm**.

### Công thức chung "DP trên trạng thái"

1. Xác định trạng thái **đủ thông tin** để quyết định tương lai: ở đây `(pos, k)`.
2. Viết chuyển trạng thái: `(pos, k) → (pos + step, step)` với `step ∈ {k-1, k, k+1}`, `step > 0`, `pos + step` là đá.
3. Chặn số trạng thái để biết Big-O: `k ≤ chỉ số đá` → `O(n²)`.
4. Chọn cách duyệt: top-down memo (dễ viết từ đệ quy), bottom-up (không đệ quy), BFS (có thêm số bước ít nhất).

### Bài liên quan

| Bài | Liên hệ |
|---|---|
| 70. Climbing Stairs | DP vị trí đơn giản (trạng thái chỉ là vị trí) |
| 55. Jump Game / 45. Jump Game II | Có tới được / ít bước nhất, mỗi ô cho tầm nhảy tối đa |
| 1306. Jump Game III, 1345. Jump Game IV | BFS trên vị trí với luật nhảy khác nhau |
| 1871. Jump Game VII | Tới được không, với khoảng nhảy `[minJump, maxJump]` |
| 2498. Frog Jump II | Ếch đi rồi quay lại, tối thiểu bước dài nhất (tham lam) |
| 1824. Minimum Sideway Jumps | DP trên (vị trí, làn đường) |
| 818. Race Car | BFS trên trạng thái (vị trí, tốc độ): cùng tinh thần "trạng thái có thêm vận tốc" |

### Biến thể tự nghĩ (sẽ gặp trong bài tập)

- **Đếm số cách** qua sông: memo trả về **số** thay vì `bool`, cộng thay vì `or`.
- **Số lần nhảy ít nhất**: BFS theo lớp.
- **DP mảng 2 chiều theo chỉ số**: `dp[i][k]` = tới được đá chỉ số `i` với bước cuối `k` (`k ≤ i`); từ đá `j < i`, `k = stones[i] - stones[j]` hợp lệ nếu `dp[j][k-1] or dp[j][k] or dp[j][k+1]`. Cũng `O(n²)`.
- **Luật nhảy khác**: `k-2..k+2`, bước đầu dài `d`... Chỉ cần đổi danh sách `step` và điểm xuất phát.

## 14. Bài tập

Làm theo thứ tự từ dễ tới khó. **Tự làm trước**, sau đó mới xem gợi ý (mục 15) và đáp án (mục 16, cuối notebook).

### Bài 1 — Đoán output (không chạy code)

Với mỗi bộ đá, `canCross` trả `True` hay `False`? Giải thích ngắn gọn.

```text
a) [0, 1]
b) [0, 2]
c) [0, 1, 3, 6, 10, 15, 21]
d) [0, 1, 3, 6, 7]
e) [0, 1, 2, 3, 6, 9]
f) [0, 1, 3, 4, 5, 7, 9, 10, 12]
```

### Bài 2 — Tính tay bảng DP

Lập bảng `dp[s]` (tập các bước `k` đáp xuống đá `s`) giống mục 7.3 cho:

```text
stones = [0, 1, 2, 3, 5, 6, 9]
```

Đáp án là gì? Đá nào có **hai** giá trị `k`? Có bao nhiêu trạng thái `(s, k)` tất cả?

### Bài 3 — Điền chỗ trống: DP bottom-up

Hoàn thiện hàm `exercise_can_cross_dp` theo các TODO (cách 3).

In [33]:
def exercise_can_cross_dp(stones: List[int]) -> bool:
    # TODO 1: tạo dp = dict, mỗi đá -> một set rỗng riêng
    # TODO 2: dp[0] chứa bước 0 (mẹo xuất phát)
    # TODO 3: duyệt đá s từ trái sang phải, với mỗi k trong dp[s]
    # TODO 4: thử step trong k-1, k, k+1; nếu step > 0 và s+step là đá -> thêm step vào dp[s+step]
    # TODO 5: trả True nếu dp của đá cuối khác rỗng
    pass

# Tự làm xong rồi mới mở test:
# assert exercise_can_cross_dp([0, 1, 3, 5, 6, 8, 12, 17]) == True
# assert exercise_can_cross_dp([0, 1, 2, 3, 4, 8, 9, 11]) == False
# assert exercise_can_cross_dp([0, 2]) == False

### Bài 4 — Tự viết lại: top-down memo + cắt tỉa

Viết `exercise_can_cross_memo(stones)` **không nhìn lại** mục 6 và 9:

- Có bước cắt tỉa `stones[i] - stones[i-1] > i → False`.
- Đệ quy `can(pos, k)` với memo theo **cặp** `(pos, k)` (tự dùng `dict` hoặc `lru_cache`).
- Xuất phát đúng luật bước đầu tiên.

In [34]:
def exercise_can_cross_memo(stones: List[int]) -> bool:
    # TODO: cắt tỉa khe > i
    # TODO: stone_set, last, memo
    # TODO: def can(pos, k): base case, thử 3 bước (step > 0), ghi sổ
    # TODO: return can(0, 0)
    raise NotImplementedError

# Tự làm xong rồi mới mở test:
# assert exercise_can_cross_memo([0, 1, 3, 5, 6, 8, 12, 17]) == True
# assert exercise_can_cross_memo([0, 1, 2, 3, 4, 8, 9, 11]) == False
# assert exercise_can_cross_memo([0, 1, 2, 3, 6, 9]) == True
# assert exercise_can_cross_memo([t * (t + 1) // 2 for t in range(2000)]) == True

### Bài 5 — Biến thể: đếm số cách qua sông

Hai cách qua sông khác nhau nếu **dãy độ dài bước nhảy** khác nhau. Viết `count_ways(stones)` trả về **số cách** đáp xuống đá cuối.

Ví dụ:

```text
count_ways([0, 1, 2, 3])        -> 2    # [1,1,1] và [1,2]
count_ways([0, 1, 3, 5, 6, 8, 12, 17]) -> 1
count_ways([0, 1, 2, 3, 4, 8, 9, 11])  -> 0
count_ways([0, 1, 2, 3, 4, 5, 6])      -> 9
```

In [35]:
def count_ways(stones: List[int]) -> int:
    # TODO: memo theo (pos, k), nhưng trả về SỐ cách thay vì True/False
    # Gợi ý: base case pos == last -> 1; cộng kết quả của các bước hợp lệ
    pass

# Tự làm xong rồi mới mở test:
# assert count_ways([0, 1, 2, 3]) == 2
# assert count_ways([0, 1, 3, 5, 6, 8, 12, 17]) == 1
# assert count_ways([0, 1, 2, 3, 4, 8, 9, 11]) == 0
# assert count_ways([0, 1, 2, 3, 4, 5, 6]) == 9

### Bài 6 — Mini challenge: số lần nhảy ít nhất và đường đi

Viết `min_jumps_path(stones)` trả về **danh sách vị trí** ếch đáp xuống (bắt đầu bằng `0`, kết thúc bằng đá cuối) theo đường có **ít lần nhảy nhất**; nếu không qua được trả `[]`.

Ví dụ:

```text
min_jumps_path([0, 1, 3, 5, 6, 8, 12, 17]) -> [0, 1, 3, 5, 8, 12, 17]   # 6 lần nhảy
min_jumps_path([0, 1, 2, 3, 4, 8, 9, 11])  -> []
min_jumps_path([0, 1, 2, 3, 4, 5, 6])      -> [0, 1, 3, 6]   # 3 lần nhảy: 1, 2, 3
```

Yêu cầu thêm: không dùng đệ quy.

In [36]:
def min_jumps_path(stones: List[int]) -> List[int]:
    # TODO: BFS trên trạng thái (pos, k)
    # TODO: lưu parent[(pos, k)] = trạng thái trước để dựng lại đường đi
    # TODO: lần đầu pop ra pos == last -> lần ngược parent, đảo lại danh sách vị trí
    pass

# Tự làm xong rồi mới mở test:
# assert min_jumps_path([0, 1, 3, 5, 6, 8, 12, 17]) == [0, 1, 3, 5, 8, 12, 17]
# assert min_jumps_path([0, 1, 2, 3, 4, 8, 9, 11]) == []
# assert min_jumps_path([0, 1, 2, 3, 4, 5, 6]) == [0, 1, 3, 6]

## 15. Gợi ý theo mức

**Bài 1**
- Mức 1: kiểm tra cắt tỉa (khe `> i`?) trước, rồi thử nhảy bằng tay.
- Mức 2: với d), tới `6` bằng bước nào? Từ đó bước tiếp được những độ dài nào?
- Mức 3: f) thử đường `1, 2, 1, 1, 2, 2, 1, 2`... hay có đường nào khác? Dùng bảng DP nếu rối.

**Bài 2**
- Mức 1: khởi đầu `dp[0] = {0}`, duyệt đá từ trái sang phải.
- Mức 2: ở mỗi đá, liệt kê mọi `k`, rồi thử `k-1, k, k+1`.
- Mức 3: đá `3` được tới từ đá `2` (bước 1) **và** từ đá `1` (bước 2).

**Bài 3**
- Mức 1: chép khung mục 7.1.
- Mức 2: `dp = {s: set() for s in stones}`; kiểm tra đá bằng `s + step in dp`.
- Mức 3: `return len(dp[stones[-1]]) > 0`.

**Bài 4**
- Mức 1: viết brute force trước, rồi thêm 3 dòng memo.
- Mức 2: khóa `(pos, k)`; xuất phát `can(0, 0)`.
- Mức 3: bộ tam giác 2000 đá cần `sys.setrecursionlimit(10000)`.

**Bài 5**
- Mức 1: "có tới được không" dùng `or`; "bao nhiêu cách" dùng `+`.
- Mức 2: `ways(pos, k) = Σ ways(pos + step, step)` trên các bước hợp lệ; `ways(last, ·) = 1`.
- Mức 3: memo vẫn đúng vì số cách từ `(pos, k)` chỉ phụ thuộc `(pos, k)`.

**Bài 6**
- Mức 1: BFS duyệt theo số lần nhảy tăng dần, nên lần đầu chạm đá cuối là ít nhất.
- Mức 2: khi đưa `(nxt, step)` vào hàng, ghi `parent[(nxt, step)] = (pos, k)`.
- Mức 3: dựng đường: từ trạng thái đích, lần theo `parent` tới `(0, 0)`, gom `pos`, rồi đảo ngược.

## 16. Đáp án tham khảo

Nên tự làm trước rồi mới xem phần này.

### Đáp án Bài 1

| Bộ đá | Kết quả | Lý do |
|---|:---:|---|
| a) `[0, 1]` | `True` | bước đầu dài 1 là tới |
| b) `[0, 2]` | `False` | bước đầu bắt buộc 1, vị trí 1 là nước (cắt tỉa: khe 2 > 1) |
| c) `[0, 1, 3, 6, 10, 15, 21]` | `True` | nhảy `1, 2, 3, 4, 5, 6`, mỗi bước tăng 1 |
| d) `[0, 1, 3, 6, 7]` | `False` | đường duy nhất tới `6` là `1, 2, 3` nên `k = 3`; bước tiếp `2, 3, 4` → `8, 9, 10`, không tới `7` |
| e) `[0, 1, 2, 3, 6, 9]` | `True` | `0→1→3→6→9` bằng `1, 2, 3, 3` |
| f) `[0, 1, 3, 4, 5, 7, 9, 10, 12]` | `True` | ví dụ đường `0→1→3→5→7→9→12` bằng `1, 2, 2, 2, 2, 3` (bảng DP bên dưới còn cho thấy đá cuối tới được bằng cả `k = 2` lẫn `k = 3`) |

### Đáp án Bài 2

`True`; `dp = {0:{0}, 1:{1}, 2:{1}, 3:{1,2}, 5:{2}, 6:{1,3}, 9:{3}}`. Hai đá có hai giá trị `k` là `3` (tới từ đá 2 bằng bước 1, từ đá 1 bằng bước 2) và `6` (từ đá 5 bằng bước 1, từ đá 3 bằng bước 3). Tổng cộng `9` trạng thái `(s, k)`. Đường qua sông duy nhất: `0→1→3→6→9` bằng `1, 2, 3, 3`.

Cell dưới kiểm chứng Bài 1 (kèm bảng DP của câu f), rồi in bảng DP của Bài 2 để so với bảng tự tính.

In [37]:
quiz = {
    "a": [0, 1], "b": [0, 2], "c": [0, 1, 3, 6, 10, 15, 21],
    "d": [0, 1, 3, 6, 7], "e": [0, 1, 2, 3, 6, 9], "f": [0, 1, 3, 4, 5, 7, 9, 10, 12],
}
for key, stones in quiz.items():
    print(key, stones, "->", Solution().canCross(stones))
print()
print("Bài 1f — bảng DP:")
trace_dp(quiz["f"])
print()
print("Bài 2 — bảng DP:")
stones_b2 = [0, 1, 2, 3, 5, 6, 9]
trace_dp(stones_b2)
ks_b2 = reachable_ks(stones_b2)
print("Đá có hai giá trị k:", [s for s in stones_b2 if len(ks_b2[s]) >= 2])
print("Tổng số trạng thái (s, k):", sum(len(v) for v in ks_b2.values()))

a [0, 1] -> True
b [0, 2] -> False
c [0, 1, 3, 6, 10, 15, 21] -> True
d [0, 1, 3, 6, 7] -> False
e [0, 1, 2, 3, 6, 9] -> True
f [0, 1, 3, 4, 5, 7, 9, 10, 12] -> True

Bài 1f — bảng DP:
đá  0 | dp = [0]     | đẩy: 0+1→1(k=0)
đá  1 | dp = [1]     | đẩy: 1+2→3(k=1)
đá  3 | dp = [2]     | đẩy: 3+1→4(k=2), 3+2→5(k=2)
đá  4 | dp = [1]     | đẩy: 4+1→5(k=1)
đá  5 | dp = [1, 2]  | đẩy: 5+2→7(k=1), 5+2→7(k=2)
đá  7 | dp = [2]     | đẩy: 7+2→9(k=2), 7+3→10(k=2)
đá  9 | dp = [2]     | đẩy: 9+1→10(k=2), 9+3→12(k=2)
đá 10 | dp = [1, 3]  | đẩy: 10+2→12(k=1), 10+2→12(k=3)
đá 12 | dp = [2, 3]  | đẩy: —
=> dp[last] = [2, 3] -> True

Bài 2 — bảng DP:
đá  0 | dp = [0]     | đẩy: 0+1→1(k=0)
đá  1 | dp = [1]     | đẩy: 1+1→2(k=1), 1+2→3(k=1)
đá  2 | dp = [1]     | đẩy: 2+1→3(k=1)
đá  3 | dp = [1, 2]  | đẩy: 3+2→5(k=1), 3+2→5(k=2), 3+3→6(k=2)
đá  5 | dp = [2]     | đẩy: 5+1→6(k=2)
đá  6 | dp = [1, 3]  | đẩy: 6+3→9(k=3)
đá  9 | dp = [3]     | đẩy: —
=> dp[last] = [3] -> True
Đá có hai giá trị k: [3, 6]
Tổng 

### Đáp án Bài 3 → 6

Mỗi hàm đặt tên `answer_...` và có assert chạy được. Bài 4 dùng `dict` tự quản lý; Bài 5 là memo trả về số; Bài 6 là BFS kèm `parent`.

In [38]:
def answer_can_cross_dp(stones: List[int]) -> bool:
    dp = {s: set() for s in stones}
    dp[0].add(0)
    for s in stones:
        for k in dp[s]:
            for step in (k - 1, k, k + 1):
                if step > 0 and s + step in dp:
                    dp[s + step].add(step)
    return len(dp[stones[-1]]) > 0


def answer_can_cross_memo(stones: List[int]) -> bool:
    for i in range(1, len(stones)):
        if stones[i] - stones[i - 1] > i:
            return False
    stone_set, last, memo = set(stones), stones[-1], {}

    def can(pos, k):
        if (pos, k) in memo:
            return memo[(pos, k)]
        if pos == last:
            return True
        result = False
        for step in (k - 1, k, k + 1):
            if step > 0 and pos + step in stone_set and can(pos + step, step):
                result = True
                break
        memo[(pos, k)] = result
        return result

    return can(0, 0)


def answer_count_ways(stones: List[int]) -> int:
    stone_set, last = set(stones), stones[-1]

    @lru_cache(maxsize=None)
    def ways(pos, k):
        if pos == last:
            return 1
        total = 0
        for step in (k - 1, k, k + 1):
            if step > 0 and pos + step in stone_set:
                total += ways(pos + step, step)        # cộng thay vì or
        return total

    return ways(0, 0)


def answer_min_jumps_path(stones: List[int]) -> List[int]:
    stone_set, last = set(stones), stones[-1]
    start = (0, 0)
    parent = {start: None}                             # cũng đóng vai trò visited
    q = deque([start])
    while q:
        state = q.popleft()
        pos, k = state
        if pos == last:
            path = []
            while state is not None:                   # lần ngược về (0, 0)
                path.append(state[0])
                state = parent[state]
            return path[::-1]
        for step in (k - 1, k, k + 1):
            nxt = (pos + step, step)
            if step > 0 and pos + step in stone_set and nxt not in parent:
                parent[nxt] = state
                q.append(nxt)
    return []


assert answer_can_cross_dp([0, 1, 3, 5, 6, 8, 12, 17]) == True
assert answer_can_cross_dp([0, 1, 2, 3, 4, 8, 9, 11]) == False
assert answer_can_cross_dp([0, 2]) == False

assert answer_can_cross_memo([0, 1, 3, 5, 6, 8, 12, 17]) == True
assert answer_can_cross_memo([0, 1, 2, 3, 4, 8, 9, 11]) == False
assert answer_can_cross_memo([0, 1, 2, 3, 6, 9]) == True
assert answer_can_cross_memo([t * (t + 1) // 2 for t in range(2000)]) == True

assert answer_count_ways([0, 1, 2, 3]) == 2
assert answer_count_ways([0, 1, 3, 5, 6, 8, 12, 17]) == 1
assert answer_count_ways([0, 1, 2, 3, 4, 8, 9, 11]) == 0
assert answer_count_ways([0, 1, 2, 3, 4, 5, 6]) == 9

assert answer_min_jumps_path([0, 1, 3, 5, 6, 8, 12, 17]) == [0, 1, 3, 5, 8, 12, 17]
assert answer_min_jumps_path([0, 1, 2, 3, 4, 8, 9, 11]) == []
assert answer_min_jumps_path([0, 1, 2, 3, 4, 5, 6]) == [0, 1, 3, 6]
print("min_jumps_path([0..6]) =", answer_min_jumps_path([0, 1, 2, 3, 4, 5, 6]))

# Đối chiếu chéo đáp án với các lời giải chính trên bộ ngẫu nhiên
random.seed(7)
for _ in range(1000):
    pos, stones = 0, [0]
    for _ in range(random.randint(1, 12)):
        pos += random.randint(1, 3)
        stones.append(pos)
    ok = can_cross_dp(stones)
    assert answer_can_cross_dp(stones) == ok
    assert answer_can_cross_memo(stones) == ok
    assert (answer_count_ways(stones) > 0) == ok
    path = answer_min_jumps_path(stones)
    assert (len(path) > 0) == ok
    if path:
        assert is_valid_jumps(stones, [b - a for a, b in zip(path, path[1:])])
print("Tất cả đáp án PASS (kể cả 1000 bộ ngẫu nhiên)!")

min_jumps_path([0..6]) = [0, 1, 3, 6]
Tất cả đáp án PASS (kể cả 1000 bộ ngẫu nhiên)!


## 17. Template nên nhớ

**Top-down (cách chính):**

```python
class Solution:
    def canCross(self, stones: List[int]) -> bool:
        for i in range(1, len(stones)):                 # cắt tỉa: bước thứ t <= t
            if stones[i] - stones[i - 1] > i:
                return False
        sys.setrecursionlimit(10000)
        stone_set, last = set(stones), stones[-1]

        @lru_cache(maxsize=None)
        def can(pos, k):                                # trạng thái = (vị trí, bước cuối)
            if pos == last:
                return True
            for step in (k + 1, k, k - 1):
                if step > 0 and pos + step in stone_set and can(pos + step, step):
                    return True
            return False

        return can(0, 0)                                # k = 0 -> bước đầu chỉ còn 1
```

**Bottom-up (không đệ quy):** `dp = {s: set()}`, `dp[0] = {0}`, duyệt đá trái → phải, với mỗi `k ∈ dp[s]` đẩy `step ∈ {k-1, k, k+1}, step > 0` vào `dp[s + step]` nếu là đá; trả `len(dp[last]) > 0`.

**Câu thần chú:** *"Vị trí thôi chưa đủ: trạng thái là (đá, bước cuối). Mỗi trạng thái tính một lần → O(n²). Bước thứ t không quá t → khe lớn hơn chỉ số thì thua ngay."*

## 18. Checklist tự kiểm tra trước khi submit

- Trạng thái (và khóa memo) là **cặp `(pos, k)`**, không phải chỉ `pos`?
- Có điều kiện **`step > 0`** chưa (tránh đứng yên / nhảy lùi / đệ quy vô hạn / lỗi set đổi kích thước)?
- Bước đầu tiên đúng luật: xuất phát `can(0, 0)` (hoặc kiểm tra `stones[1] == 1`), **không** phải `can(0, 1)`?
- Tra "có phải đá không" bằng **`set`/`dict`**, không phải `list`?
- Bản đệ quy: đã `sys.setrecursionlimit(10000)` (chuỗi nhảy có thể dài ~2000)? `lru_cache` đặt **bên trong** method?
- Bản bottom-up: duyệt đá theo thứ tự **tăng dần**, trả về **`bool`** chứ không phải `set`?
- Cắt tỉa `stones[i] - stones[i-1] > i → False` đặt đúng chỗ, và vẫn chạy thuật toán đầy đủ khi không bị cắt?
- Đã test: 2 ví dụ của đề, `[0, 1]`, `[0, 2]`, `[0, 1, 3, 6, 7]`, `[0, 1, 2, 3, 6, 9]` (bẫy memo theo vị trí), vị trí `2^31 - 1`, bộ tam giác 2000 đá và bộ 2000 đá phải duyệt hết?